# 99.0 Reproduce full pipeline

## Notebook overview
This notebook reproduces the full end-to-end anomaly detection pipeline for the MVTec AD dataset in one place. It covers dataset auditing, split creation, exact leakage checking, baseline ImageNet models, SimCLR pretraining, the matched final comparison, threshold analysis, ablations, failure analysis, and final report-style figures. The notebook is designed as a standalone full-run notebook for Kaggle, while also remaining usable in a local environment if the dataset path is set correctly.

## Purpose
- reproduce the entire anomaly detection study from one notebook
- keep the full workflow in a single, readable place for GitHub and reproducibility
- save clean artefacts including figures, tables, checkpoints, and JSON metadata
- support both a quick debug mode and a final full run mode


In [ ]:
#------------------------------------------------------------------------------
# 1.0 Install deps
#------------------------------------------------------------------------------
# Only install packages if they are missing in the current runtime.
import importlib
import subprocess
import sys

MISSING_PACKAGES = []
for package_name, import_name in [("faiss-cpu", "faiss"), ("tqdm", "tqdm")]:
    try:
        importlib.import_module(import_name)
    except Exception:
        MISSING_PACKAGES.append(package_name)

if MISSING_PACKAGES:
    print("Installing missing packages:", MISSING_PACKAGES)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *MISSING_PACKAGES])
else:
    print("Required packages already available.")


In [ ]:
#------------------------------------------------------------------------------
# 1.1 Imports + global helpers + runtime environment
#------------------------------------------------------------------------------
import os, sys, json, time, math, random, hashlib, shutil, subprocess, textwrap, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

import faiss
from tqdm.auto import tqdm

#------------------------------------------------------------------------------
# Small notebook helpers
#------------------------------------------------------------------------------
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)

def now_stamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def write_json_overwrite(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def read_json(path):
    with open(Path(path), "r") as f:
        return json.load(f)

def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan

def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def get_peak_gpu_memory_mb():
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()

print_banner("Runtime environment")
print("Kaggle runtime :", IS_KAGGLE)
print("Python         :", sys.version.split()[0])
print("Torch          :", torch.__version__)
print("Torchvision    :", torchvision.__version__)
print("CUDA available :", torch.cuda.is_available())
print("GPU count      :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} name     :", torch.cuda.get_device_name(i))


# 2.0 Run settings

## Purpose
- define the main experimental settings in one place before any data processing or modelling begins
- make it easy to switch between a smaller debug run and a full run without changing the rest of the notebook structure
- keep the most important paths, hyperparameters, category settings, and evaluation controls visible near the top of the notebook
- make the later sections easier to follow by centralising values that are reused throughout training, evaluation, thresholding, and visualisation
- support reproducibility by keeping the active configuration fixed and easy to inspect during marking

## Process
1. set the run mode and choose the active categories for the current experiment.
2. define the key training and evaluation settings used later in the notebook.
3. create the main folders for saved tables, figures, checkpoints, and summaries.
4. lock the shared comparison settings so the final results are evaluated fairly.

In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core run settings
#------------------------------------------------------------------------------
NOTEBOOK_ID = "mvtec_full_pipeline_hardened"
RUN_MODE = os.environ.get("MVTEC_RUN_MODE", "full")
SEED = 42
DETERMINISTIC_DEBUG = False
set_seed(SEED, deterministic=(RUN_MODE == "debug" and DETERMINISTIC_DEBUG))

DEBUG_CATEGORIES = ["carpet", "tile", "bottle", "transistor"]
CATEGORIES_ALL = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]
CATEGORIES = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL
QUAL_CATEGORY = next((c for c in ["carpet", "tile", "bottle", "transistor"] if c in CATEGORIES), CATEGORIES[0])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()

#------------------------------------------------------------------------------
# Output folders
#------------------------------------------------------------------------------
DEFAULT_WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
WORK_ROOT = Path(os.environ.get("MVTEC_WORK_ROOT", str(DEFAULT_WORK_ROOT / "mvtec_unified_project")))
ARTIFACTS_DIR = ensure_dir(WORK_ROOT / "artifacts" / RUN_MODE)
RESULTS_DIR = ensure_dir(WORK_ROOT / "results" / RUN_MODE)
CHECKPOINTS_DIR = ensure_dir(WORK_ROOT / "checkpoints" / RUN_MODE)

for folder_name in ["tables", "figures", "json", "samples", "logs"]:
    ensure_dir(RESULTS_DIR / folder_name)

CONFIG_JSON_PATH = ARTIFACTS_DIR / "config_run.json"

#------------------------------------------------------------------------------
# General image/data settings
#------------------------------------------------------------------------------
IMG_SIZE = 224
VAL_FRAC = 0.10
CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS_BASE = min(4, max(2, CPU_COUNT // 2))
NUM_WORKERS = NUM_WORKERS_BASE if DEVICE == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

#------------------------------------------------------------------------------
# Baseline settings
#------------------------------------------------------------------------------
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_TEST = 16

BACKBONE_NAME = "resnet18_imagenet"
PATCH_LAYER_NAME_MAIN = "layer3"
PATCH_LAYER_NAMES = [PATCH_LAYER_NAME_MAIN]
PADIM_LAYER_NAME = PATCH_LAYER_NAME_MAIN

CORESET_RATIO = 0.05
MAX_TRAIN_PATCHES = 120_000
PATCH_SCORE_TOPK = 200

PADIM_DIM = 100
PADIM_EPS = 1e-4

AE_EPOCHS = 3 if RUN_MODE == "debug" else 5
AE_LR = 1e-3

#------------------------------------------------------------------------------
# SimCLR settings
#------------------------------------------------------------------------------
SIMCLR_BATCH_SIZE_SINGLE = 128 if DEVICE == "cuda" else 32
SIMCLR_EPOCHS = 10 if RUN_MODE == "debug" else 30
SIMCLR_LR = 3e-4
SIMCLR_WEIGHT_DECAY = 1e-4
SIMCLR_TEMPERATURE = 0.2
SIMCLR_PROJ_DIM = 128

SIMCLR_AUG_STRENGTHS = ["strong"] if RUN_MODE == "debug" else ["mild", "strong"]

#------------------------------------------------------------------------------
# Evaluation settings
#------------------------------------------------------------------------------
MAIN_METHODS = [
    "imagenet_patchcore",
    "imagenet_padim",
    "autoencoder",
    "simclr_mild_patchcore",
    "simclr_strong_patchcore",
    "simclr_mild_padim",
    "simclr_strong_padim",
]

THRESHOLD_POLICIES = ["high_recall", "balanced", "low_false_alarm"]
DENSE_SWEEP_POINTS = 40 if RUN_MODE == "debug" else 120

print_banner("Core run settings")
print("NOTEBOOK_ID :", NOTEBOOK_ID)
print("RUN_MODE    :", RUN_MODE)
print("DEVICE      :", DEVICE)
print("GPU_COUNT   :", GPU_COUNT)
print("WORK_ROOT   :", WORK_ROOT)
print("CATEGORIES  :", CATEGORIES)

write_json_overwrite(
    {
        "NOTEBOOK_ID": NOTEBOOK_ID,
        "RUN_MODE": RUN_MODE,
        "SEED": SEED,
        "DEVICE": DEVICE,
        "GPU_COUNT": GPU_COUNT,
        "WORK_ROOT": str(WORK_ROOT),
        "MVTEC_WORK_ROOT": str(WORK_ROOT),
        "IMG_SIZE": IMG_SIZE,
        "VAL_FRAC": VAL_FRAC,
        "PATCH_LAYER_NAME_MAIN": PATCH_LAYER_NAME_MAIN,
        "CORESET_RATIO": CORESET_RATIO,
        "PADIM_DIM": PADIM_DIM,
        "AE_EPOCHS": AE_EPOCHS,
        "SIMCLR_EPOCHS": SIMCLR_EPOCHS,
        "SIMCLR_BATCH_SIZE_SINGLE": SIMCLR_BATCH_SIZE_SINGLE,
        "SIMCLR_LR": SIMCLR_LR,
        "SIMCLR_TEMPERATURE": SIMCLR_TEMPERATURE,
        "SIMCLR_AUG_STRENGTHS": SIMCLR_AUG_STRENGTHS,
        "THRESHOLD_POLICIES": THRESHOLD_POLICIES,
        "DENSE_SWEEP_POINTS": DENSE_SWEEP_POINTS,
        "CATEGORIES": CATEGORIES,
    },
    CONFIG_JSON_PATH,
)


# 3.0 Set Kaggle dataset path

## Purpose
- define the dataset location used by the notebook inside the Kaggle runtime

In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve dataset path
#------------------------------------------------------------------------------
DATASET_CANDIDATES = []

env_mvtec_dir = os.environ.get("MVTEC_DIR")
if env_mvtec_dir:
    DATASET_CANDIDATES.append(Path(env_mvtec_dir))

if IS_KAGGLE:
    DATASET_CANDIDATES.extend([
        Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
        Path("/kaggle/input/mvtec-ad"),
    ])

DATASET_CANDIDATES.extend([
    Path.cwd() / "mvtec-ad",
    Path.cwd() / "data" / "mvtec-ad",
    WORK_ROOT / "data" / "mvtec-ad",
])

MVTEC_DIR = next((p for p in DATASET_CANDIDATES if p.exists()), None)

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "Could not find the MVTec AD dataset. Set the MVTEC_DIR environment variable "
        "or place the dataset in one of the expected locations."
    )

category_dirs = sorted([p.name for p in MVTEC_DIR.iterdir() if p.is_dir()])

print_banner("Dataset root")
print("MVTEC_DIR:", MVTEC_DIR)
print("Top-level categories found:", category_dirs[:15])

missing_categories = [c for c in CATEGORIES if c not in category_dirs]
if missing_categories:
    raise ValueError(f"Missing expected categories in MVTEC_DIR: {missing_categories}")


# 4.0 Dataset audit

## Purpose
- inspect the structure of the MVTec AD dataset before training begins
- confirm that the expected category folders, train folders, test folders, and ground-truth mask folders are present
- summarise the number of normal training images, test images, anomaly images, and masks by category
- check whether mask naming and sampled image-mask pairs look consistent enough for later localisation evaluation
- save simple overview figures and summary tables that can also support the dataset section of the report

## Process
1. scan each category folder and record how many train, test, anomaly, and mask files are available.
2. check file naming patterns and sample image-mask pairs for basic consistency.
3. save audit tables and overview figures so the dataset characteristics are documented before modelling starts.

In [ ]:
#------------------------------------------------------------------------------
# 4.1 Audit helpers
#------------------------------------------------------------------------------

def list_pngs(folder_path):
    folder_path = Path(folder_path)
    return sorted(folder_path.glob("*.png")) if folder_path.exists() else []

# List the immediate subfolders for a category split.
def list_subdirs(folder_path):
    folder_path = Path(folder_path)
    return sorted([p for p in folder_path.iterdir() if p.is_dir()]) if folder_path.exists() else []

# Grab a small sample of items for quick visual checks.
def pick_n(items, n):
    if len(items) == 0:
        return []
    idx = np.random.choice(len(items), size=min(n, len(items)), replace=False)
    return [items[i] for i in idx]

# Read an RGB image into a NumPy array for plotting.
def load_rgb_np(path):
    return np.array(Image.open(path).convert("RGB"))

# Read a mask image into a NumPy array.
def load_mask_np(path):
    return np.array(Image.open(path).convert("L"))

# Standardise mask names so masks and images can be matched.
def mask_stem(mask_path):
    s = Path(mask_path).stem
    return s[:-5] if s.endswith("_mask") else s

# Build a quick lookup from image stem to mask path.
def build_mask_lookup(mask_paths):
    return {mask_stem(p): p for p in mask_paths}

# Inspect mask naming so later matching is consistent.
def infer_mask_style(mask_paths):
    n_mask_suffix = sum(Path(p).stem.endswith("_mask") for p in mask_paths)
    return "stem_mask" if n_mask_suffix > 0 else "same_name"

# Pick one defect folder for the example visualisations.
def choose_example_defect(cat_dir):
    candidates = []
    for defect_dir in list_subdirs(cat_dir / "test"):
        defect = defect_dir.name
        if defect == "good":
            continue
        img_paths = list_pngs(cat_dir / "test" / defect)
        mask_paths = list_pngs(cat_dir / "ground_truth" / defect)
        img_stems = set(p.stem for p in img_paths)
        mask_stems = set(mask_stem(p) for p in mask_paths)
        paired_n = len(img_stems.intersection(mask_stems))
        if paired_n > 0:
            candidates.append((paired_n, defect, img_paths, mask_paths))
    if len(candidates) == 0:
        return None, [], {}
    candidates = sorted(candidates, key=lambda x: (-x[0], x[1]))
    _, defect, img_paths, mask_paths = candidates[0]
    return defect, img_paths, build_mask_lookup(mask_paths)

#------------------------------------------------------------------------------
# 4.2 Build dataset audit table
#------------------------------------------------------------------------------
rows = []
SAMPLE_SHAPE_CHECK_PER_DEFECT = 5

# Audit every category, even in debug mode, so the dataset summary stays complete.
for cat in CATEGORIES_ALL:
    cat_dir = MVTEC_DIR / cat
    train_dir = cat_dir / "train"
    test_dir = cat_dir / "test"
    gt_dir = cat_dir / "ground_truth"

    train_subdirs = [d.name for d in list_subdirs(train_dir)]
    train_only_good = int(set(train_subdirs) == {"good"})

    train_good_paths = list_pngs(train_dir / "good")
    test_good_paths = list_pngs(test_dir / "good")
    defect_types = [d.name for d in list_subdirs(test_dir) if d.name != "good"]

    test_anomaly_total, masks_total = 0, 0
    mask_missing_total, mask_extra_total = 0, 0
    shape_mismatch_sample = 0
    style_votes = {"same_name": 0, "stem_mask": 0}

    # Count anomaly images and check whether the masks line up with them.
    for defect in defect_types:
        defect_img_paths = list_pngs(test_dir / defect)
        defect_mask_paths = list_pngs(gt_dir / defect)

        test_anomaly_total += len(defect_img_paths)
        masks_total += len(defect_mask_paths)

        style = infer_mask_style(defect_mask_paths)
        style_votes[style] += 1

        img_stems = set(p.stem for p in defect_img_paths)
        mask_stems = set(mask_stem(p) for p in defect_mask_paths)

        mask_missing_total += len(img_stems - mask_stems)
        mask_extra_total += len(mask_stems - img_stems)

        if len(defect_img_paths) > 0 and len(defect_mask_paths) > 0:
            mask_lookup = build_mask_lookup(defect_mask_paths)
            sample_imgs = defect_img_paths[:SAMPLE_SHAPE_CHECK_PER_DEFECT]

            for img_path in sample_imgs:
                mask_path = mask_lookup.get(img_path.stem, None)
                if mask_path is None:
                    continue
                img = np.array(Image.open(img_path))
                msk = np.array(Image.open(mask_path))
                if img.shape[0] != msk.shape[0] or img.shape[1] != msk.shape[1]:
                    shape_mismatch_sample += 1

    mask_style = "stem_mask" if style_votes["stem_mask"] > style_votes["same_name"] else "same_name"

    rows.append({
        "category": cat,
        "train_good_n": len(train_good_paths),
        "test_good_n": len(test_good_paths),
        "test_anomaly_n": test_anomaly_total,
        "test_total_n": len(test_good_paths) + test_anomaly_total,
        "n_defect_types": len(defect_types),
        "defect_types": ";".join(defect_types),
        "masks_total_n": masks_total,
        "has_masks": int(masks_total > 0),
        "train_only_good": train_only_good,
        "mask_style": mask_style,
        "mask_missing_total": mask_missing_total,
        "mask_extra_total": mask_extra_total,
        "shape_mismatch_sample": shape_mismatch_sample})

df_dataset = pd.DataFrame(rows).sort_values("category").reset_index(drop=True)
df_dataset.to_csv(TABLE_DATASET_PATH, index=False)
display(df_dataset)
print("Saved:", TABLE_DATASET_PATH)

In [ ]:
#------------------------------------------------------------------------------
# 4.3 Dataset figures
#------------------------------------------------------------------------------
cats = df_dataset["category"].tolist()
train_good_n = df_dataset["train_good_n"].values
test_good_n = df_dataset["test_good_n"].values
test_anom_n = df_dataset["test_anomaly_n"].values
masks_n = df_dataset["masks_total_n"].values
anom_ratio = test_anom_n / np.maximum(test_good_n + test_anom_n, 1)

plt.figure(figsize=(16, 10))
ax1 = plt.subplot(2, 2, 1)
ax1.bar(cats, train_good_n)
ax1.set_title("Train good images by category")
ax1.set_ylabel("Count")
ax1.tick_params(axis="x", rotation=45)

ax2 = plt.subplot(2, 2, 2)
ax2.bar(cats, test_good_n, label="Test good")
ax2.bar(cats, test_anom_n, bottom=test_good_n, label="Test anomaly")
ax2.set_title("Test images by category")
ax2.set_ylabel("Count")
ax2.tick_params(axis="x", rotation=45)
ax2.legend()

ax3 = plt.subplot(2, 2, 3)
ax3.bar(cats, masks_n)
ax3.set_title("Ground-truth masks by category")
ax3.set_ylabel("Count")
ax3.tick_params(axis="x", rotation=45)

ax4 = plt.subplot(2, 2, 4)
ax4.bar(cats, anom_ratio)
ax4.set_title("Anomaly ratio in test split")
ax4.set_ylabel("Anomaly / test total")
ax4.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(FIG1_PATH, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", FIG1_PATH)

PREFERRED_QUAL_CATS = ["carpet", "tile", "bottle", "transistor"]
QUAL_CATS = [c for c in PREFERRED_QUAL_CATS if c in CATEGORIES_ALL]
if len(QUAL_CATS) < 4:
    QUAL_CATS = CATEGORIES_ALL[:4]

N_NORMAL, N_ANOM = 3, 2
plt.figure(figsize=(3.2 * (N_NORMAL + N_ANOM), 3.2 * len(QUAL_CATS)))

for r, cat in enumerate(QUAL_CATS):
    cat_dir = MVTEC_DIR / cat
    train_good = list_pngs(cat_dir / "train" / "good")
    normal_imgs = pick_n(train_good, N_NORMAL)

    defect_name, defect_imgs_all, mask_lookup = choose_example_defect(cat_dir)
    defect_imgs = pick_n(defect_imgs_all, N_ANOM) if defect_name is not None else []

    for c in range(N_NORMAL):
        ax = plt.subplot(len(QUAL_CATS), N_NORMAL + N_ANOM, r * (N_NORMAL + N_ANOM) + c + 1)
        if c < len(normal_imgs):
            ax.imshow(load_rgb_np(normal_imgs[c]))
        ax.set_axis_off()
        if r == 0:
            ax.set_title(f"Normal {c+1}")
        if c == 0:
            ax.text(-0.08, 0.5, cat, transform=ax.transAxes, rotation=90, va="center", ha="right", fontsize=11)

    for k in range(N_ANOM):
        ax = plt.subplot(len(QUAL_CATS), N_NORMAL + N_ANOM, r * (N_NORMAL + N_ANOM) + (N_NORMAL + k) + 1)
        if k < len(defect_imgs):
            img_path = defect_imgs[k]
            ax.imshow(load_rgb_np(img_path))
            mask_path = mask_lookup.get(img_path.stem, None)
            if mask_path is not None:
                mask = load_mask_np(mask_path) > 0
                overlay = np.zeros((mask.shape[0], mask.shape[1], 4), dtype=float)
                overlay[..., 0] = 1.0
                overlay[..., 3] = mask.astype(float) * 0.35
                ax.imshow(overlay)
            ax.set_title(f"Anom {k+1}\n{defect_name}", fontsize=10)
        else:
            ax.set_title(f"Anom {k+1}", fontsize=10)
        ax.set_axis_off()

plt.tight_layout()
plt.savefig(FIG2_PATH, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", FIG2_PATH)

# 5.0 Split manifest and exact leakage checks

## Purpose
- create the shared data split structure used throughout the notebook
- separate defect-free training images into `train_good` and `val_good` so that threshold selection can be done on held-out normal data
- build a mixed test split containing both normal and anomalous images for final evaluation
- reduce the risk of overstating model performance by checking for exact overlap and exact duplicate leakage across the splits
- save a split manifest and leakage summary so the data setup is transparent and reproducible

## Process
1. split the normal training images into training and validation subsets.
2. build the mixed test set with labels, image paths, and mask paths where available.
3. run exact path-overlap checks and exact MD5 duplicate checks across the splits.
4. save the split manifest and leakage report for later reference.

In [ ]:
#------------------------------------------------------------------------------
# 5.1 Split helpers + split creation
#------------------------------------------------------------------------------
def find_mask_path(cat_dir, defect_type, img_path):
    # Good images do not have masks, so only anomaly folders need a lookup.
    if defect_type == "good":
        return None
    gt_dir = Path(cat_dir) / "ground_truth" / defect_type
    lookup = {mask_stem(p): str(p) for p in list_pngs(gt_dir)}
    return lookup.get(Path(img_path).stem, None)

# Split the normal training images into train and validation lists.
def split_train_val(paths, val_frac):
    paths = list(paths)
    idx = np.random.permutation(len(paths))
    n_val = max(1, int(round(val_frac * len(paths))))
    n_val = min(n_val, len(paths) - 1)
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]
    train_paths = [str(paths[i]) for i in train_idx]
    val_paths = [str(paths[i]) for i in val_idx]
    return train_paths, val_paths

splits = {}
split_rows = []

# Build the exact same split manifest for each active category.
for cat in CATEGORIES:
    cat_dir = MVTEC_DIR / cat
    train_good_all = list_pngs(cat_dir / "train" / "good")
    train_good, val_good = split_train_val(train_good_all, VAL_FRAC)

    # The test split keeps both good and anomalous images.
    test_items = []
    defect_types = [d.name for d in list_subdirs(cat_dir / "test")]
    for defect in defect_types:
        for p in list_pngs(cat_dir / "test" / defect):
            label = 0 if defect == "good" else 1
            test_items.append({
                "img_path": str(p),
                "label": int(label),
                "mask_path": find_mask_path(cat_dir, defect, p),
                "defect_type": defect})

    splits[cat] = {"train_good": train_good, "val_good": val_good, "test": test_items}
    split_rows.append({
        "category": cat,
        "train_good_n": len(train_good),
        "val_good_n": len(val_good),
        "test_total_n": len(test_items),
        "test_good_n": sum(int(x["label"]) == 0 for x in test_items),
        "test_anomaly_n": sum(int(x["label"]) == 1 for x in test_items),})

write_json_overwrite(splits, SPLITS_PATH)
df_splits = pd.DataFrame(split_rows).sort_values("category").reset_index(drop=True)
display(df_splits)
print("Saved:", SPLITS_PATH)

#------------------------------------------------------------------------------
# 5.2 Exact leakage checks
#------------------------------------------------------------------------------
def md5_file(path, chunk_size=8192):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# Build MD5 hashes for a list of paths.
def build_md5_map(paths, max_n=None):
    use_paths = list(paths)[:max_n] if max_n is not None else list(paths)
    return {p: md5_file(p) for p in use_paths}

# Count duplicate files inside one split.
def count_exact_md5_dupe_groups(md5_vals):
    s = pd.Series(md5_vals)
    return int((s.value_counts() > 1).sum())

# Count exact duplicate files across two different splits.
def cross_exact_md5(m1_vals, m2_vals):
    return int(len(set(m1_vals).intersection(set(m2_vals))))

leak_rows = []

# Check whether any file appears twice across train, val, and test.
for cat in CATEGORIES:
    train_paths = splits[cat]["train_good"]
    val_paths = splits[cat]["val_good"]
    test_paths = [x["img_path"] for x in splits[cat]["test"]]

    ov_train_val = len(set(train_paths).intersection(set(val_paths)))
    ov_train_test = len(set(train_paths).intersection(set(test_paths)))
    ov_val_test = len(set(val_paths).intersection(set(test_paths)))

    cap = 250 if RUN_MODE == "debug" else None
    train_md5_vals = list(build_md5_map(train_paths, max_n=cap).values())
    val_md5_vals = list(build_md5_map(val_paths, max_n=cap).values())
    test_md5_vals = list(build_md5_map(test_paths, max_n=cap).values())

    leak_rows.append({
        "category": cat,
        "path_overlap_train_val": ov_train_val,
        "path_overlap_train_test": ov_train_test,
        "path_overlap_val_test": ov_val_test,
        "train_md5_dupe_groups": count_exact_md5_dupe_groups(train_md5_vals),
        "val_md5_dupe_groups": count_exact_md5_dupe_groups(val_md5_vals),
        "test_md5_dupe_groups": count_exact_md5_dupe_groups(test_md5_vals),
        "cross_md5_train_val": cross_exact_md5(train_md5_vals, val_md5_vals),
        "cross_md5_train_test": cross_exact_md5(train_md5_vals, test_md5_vals),
        "cross_md5_val_test": cross_exact_md5(val_md5_vals, test_md5_vals),
        "md5_cap_per_split": cap})

df_leak = pd.DataFrame(leak_rows).sort_values("category").reset_index(drop=True)
leakage_report = {
    "note": "Exact duplicate checks only: path overlap + MD5 on raw file bytes. No perceptual or embedding-based near-duplicate detection.",
    "summary": {
        "total_path_overlap_train_val": int(df_leak["path_overlap_train_val"].sum()),
        "total_path_overlap_train_test": int(df_leak["path_overlap_train_test"].sum()),
        "total_path_overlap_val_test": int(df_leak["path_overlap_val_test"].sum()),
        "total_cross_md5_train_val": int(df_leak["cross_md5_train_val"].sum()),
        "total_cross_md5_train_test": int(df_leak["cross_md5_train_test"].sum()),
        "total_cross_md5_val_test": int(df_leak["cross_md5_val_test"].sum()),},
    "by_category": df_leak.to_dict(orient="records")}
write_json_overwrite(leakage_report, LEAKAGE_REPORT_PATH)
display(df_leak)
print("Saved:", LEAKAGE_REPORT_PATH)
print(leakage_report["summary"])

# 6.0 Datasets, transforms, metrics, and shared helpers

## Purpose
- define the common image transforms, dataset classes, dataloaders, and helper functions used across the notebook
- keep shared evaluation code in one place so the modelling sections can focus on the actual experiment rather than repeated setup
- make the later sections easier to read by separating reusable building blocks from the method-specific training and scoring code
- define the shared metrics for image-level anomaly detection and pixel-level localisation so the different methods can be compared fairly

In [ ]:
#------------------------------------------------------------------------------
# 6.1 Transforms + dataset + loaders
#------------------------------------------------------------------------------
# The feature methods use ImageNet normalisation, while the AE keeps raw 0-1 inputs.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

tfm_imagenet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

tfm_ae = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()])

# Load one RGB image for a dataset item.
def load_rgb(path):
    return Image.open(path).convert("RGB")

# Load one mask and convert it to a binary float array.
def load_mask(path):
    if path is None:
        return torch.zeros((1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)
    m = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    m = (np.array(m, dtype=np.float32) > 0).astype(np.float32)
    return torch.from_numpy(m)[None, :, :]

# Return images, labels, masks, and paths in a consistent dataset format.
class MvtecDataset(Dataset):
    def __init__(self, items, mode, img_tfm):
        self.items = items
        self.mode = mode
        self.img_tfm = img_tfm

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        # For train_good and val_good, the item is just an image path.
        item = self.items[i]
        if self.mode in ["train_good", "val_good"]:
            img_path, label, mask_path = item, 0, None
        else:
            img_path, label, mask_path = item["img_path"], int(item["label"]), item["mask_path"]
        x = self.img_tfm(load_rgb(img_path))
        m = load_mask(mask_path)
        return x, int(label), m, str(img_path)

# Create a DataLoader with settings matched to the current device.
def make_loader(items, mode, img_tfm, batch_size, shuffle):
    ds = MvtecDataset(items, mode=mode, img_tfm=img_tfm)
    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"))
    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(ds, **loader_kwargs)

# Build the train, validation, and test loaders for one category.
def make_split_loaders(cat, input_kind):
    img_tfm = tfm_imagenet if input_kind == "imagenet" else tfm_ae
    train_loader = make_loader(splits[cat]["train_good"], mode="train_good", img_tfm=img_tfm, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loader = make_loader(splits[cat]["val_good"], mode="val_good", img_tfm=img_tfm, batch_size=BATCH_SIZE_TEST, shuffle=False)
    test_loader = make_loader(splits[cat]["test"], mode="test", img_tfm=img_tfm, batch_size=BATCH_SIZE_TEST, shuffle=False)
    return train_loader, val_loader, test_loader

#------------------------------------------------------------------------------
# 6.2 Metrics + plotting helpers - keep the evaluation code short and make the metric handling consistent
#------------------------------------------------------------------------------
def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)

# Compute PR-AUC while handling edge cases cleanly.
def safe_pr_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_score)

# Flatten masks and heatmaps so pixel ROC-AUC can be measured.
def pixel_roc_auc(masks_list, heatmaps_list):
    y_true = np.concatenate([m.reshape(-1) for m in masks_list]).astype(int)
    y_score = np.concatenate([h.reshape(-1) for h in heatmaps_list]).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)

# Convert anomaly scores into labels and compute threshold metrics.
def threshold_metrics(y_true, y_score, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    normal_mask = (y_true == 0)
    fp = np.sum((y_pred == 1) & normal_mask)
    n_normal = np.sum(normal_mask)
    fpr = fp / max(n_normal, 1)

    return {"precision": float(precision), "recall": float(recall), "f1": float(f1), "fpr": float(fpr)}

# Rescale an array to the 0 to 1 range for plotting.
def norm_01(x):
    x = x.astype(np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

# Undo tensor formatting so images display correctly.
def tensor_to_display(img_tensor):
    img = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    if img.min() < 0 or img.max() > 1.5:
        img = img * np.array(IMAGENET_STD)[None, None, :] + np.array(IMAGENET_MEAN)[None, None, :]
    return np.clip(img, 0.0, 1.0)

# Draw an image with an optional heatmap overlay.
def overlay(ax, img_tensor, heat_2d=None, title=""):
    ax.imshow(tensor_to_display(img_tensor))
    if heat_2d is not None:
        ax.imshow(norm_01(heat_2d), alpha=0.45)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

# Show a mask with a clean grayscale colourmap.
def show_mask(ax, mask_2d, title=""):
    ax.imshow(mask_2d, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

# 7.0 Baseline methods

## Purpose
- establish strong baseline results before introducing self-supervised feature learning
- evaluate ImageNet PatchCore, ImageNet PaDiM, and the autoencoder under the same dataset split and reporting framework
- create a clear reference point so that any later SimCLR-based improvement or underperformance can be interpreted properly
- save baseline metrics, summary tables, and figures that will later be used in the main comparison and report discussion
- show performance from both detection and localisation perspectives rather than relying on only one metric

## Process
1. build the shared feature backbone and required loaders for the baseline stage.
2. run PatchCore, PaDiM, and the autoencoder for each active category.
3. collect image-level and pixel-level metrics into a baseline summary table.
4. save baseline figures so the early results are easy to inspect and compare.

In [ ]:
#------------------------------------------------------------------------------
# 7.1 Backbones + hooks + PatchCore + PaDiM + Autoencoder
#------------------------------------------------------------------------------
def get_resnet_imagenet():
    model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Identity()
    model.eval()
    return model

# Build the same ResNet encoder shape used for SimCLR checkpoints.
def get_resnet18_ssl():
    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.Identity()
    model.eval()
    return model

# Register hooks so intermediate feature maps can be captured.
def make_feature_hooks(model, layer_names):
    feats, handles = {}, []
    def hook_fn(name):
        def _hook(_, __, out):
            feats[name] = out
        return _hook
    for ln in layer_names:
        h = getattr(model, ln).register_forward_hook(hook_fn(ln))
        handles.append(h)
    return feats, handles

@torch.no_grad()
def forward_get_feats(model, feats, x, layer_names):
    _ = model(x)
    return [feats[ln] for ln in layer_names]

# Flatten a feature map into a patch-by-feature matrix.
def fmap_to_patches(fmap):
    b, c, h, w = fmap.shape
    return fmap.permute(0, 2, 3, 1).reshape(b, h * w, c)

# Upsample and concatenate multi-layer patch features.
def concat_patch_features(fm_list):
    target_h, target_w = fm_list[-1].shape[-2:]
    patch_list = []
    for fm in fm_list:
        if fm.shape[-2:] != (target_h, target_w):
            fm = F.interpolate(fm, size=(target_h, target_w), mode="bilinear", align_corners=False)
        patch_list.append(fmap_to_patches(fm))
    return torch.cat(patch_list, dim=-1)

# Build a FAISS L2 index for fast nearest-neighbour search.
def faiss_index_l2(x):
    index = faiss.IndexFlatL2(x.shape[1])
    index.add(x)
    return index

# Build the PatchCore memory bank from normal training patches.
def build_patch_bank(model, feats, train_loader, layer_names, coreset_ratio=CORESET_RATIO, max_train_patches=MAX_TRAIN_PATCHES):
    patch_chunks = []
    total = 0

    # Collect patch features from normal training images.
    for x, _, _, _ in train_loader:
        x = x.to(DEVICE)
        fm_list = forward_get_feats(model, feats, x, layer_names)
        patches = concat_patch_features(fm_list).detach().cpu().numpy()
        patches = patches.reshape(-1, patches.shape[-1]).astype(np.float32)
        patch_chunks.append(patches)
        total += patches.shape[0]

        if total >= max_train_patches:
            break

    bank = np.concatenate(patch_chunks, axis=0).astype(np.float32)

    # Keep the coreset size tied directly to the requested ratio.
    keep_n = max(1, int(round(len(bank) * float(coreset_ratio))))
    keep_n = min(keep_n, len(bank))

    idx = np.random.choice(len(bank), size=keep_n, replace=False)
    return bank[idx]

@torch.no_grad()
# Score each image from the nearest-neighbour patch distances and upsample the heatmap.
def patchcore_scores(model, feats, x, layer_names, index):
    x = x.to(DEVICE)
    fm_list = forward_get_feats(model, feats, x, layer_names)
    patches = concat_patch_features(fm_list).detach().cpu().numpy()
    b, n, d = patches.shape
    q = patches.reshape(-1, d).astype(np.float32)
    dist2, _ = index.search(q, 1)
    dist2 = dist2.reshape(b, n)
    h, w = fm_list[-1].shape[-2:]
    heat = dist2.reshape(b, h, w)
    heat_t = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(heat_t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    heat_up = heat_up.squeeze(1).numpy()
    flat = heat_up.reshape(b, -1)
    scores = np.mean(np.sort(flat, axis=1)[:, -PATCH_SCORE_TOPK:], axis=1)
    return scores, heat_up

# Fit the PaDiM Gaussian statistics from normal training patches.
def padim_fit(model, feats, train_loader, layer_name, padim_dim=PADIM_DIM, padim_eps=PADIM_EPS):
    all_fm = []
    for x, _, _, _ in train_loader:
        x = x.to(DEVICE)
        fm = forward_get_feats(model, feats, x, [layer_name])[0].detach().cpu()
        all_fm.append(fm)
    fm = torch.cat(all_fm, dim=0)
    n, c, h, w = fm.shape
    dim = min(padim_dim, c)
    idx = np.random.choice(c, size=dim, replace=False)
    fm = fm[:, idx, :, :]
    fm_np = fm.permute(0, 2, 3, 1).reshape(n, h * w, dim).numpy()
    mu = fm_np.mean(axis=0)
    cov_inv = []
    eye = np.eye(dim, dtype=np.float32)
    for i in range(h * w):
        x_i = fm_np[:, i, :]
        cov = np.cov(x_i, rowvar=False).astype(np.float32) + padim_eps * eye
        cov_inv.append(np.linalg.inv(cov).astype(np.float32))
    cov_inv = np.stack(cov_inv, axis=0)
    return {"idx": idx.astype(np.int64), "mu": mu.astype(np.float32), "cov_inv": cov_inv.astype(np.float32), "H": h, "W": w, "D": dim}

@torch.no_grad()
def padim_scores(model, feats, x, layer_name, stats):
    x = x.to(DEVICE)
    fm = forward_get_feats(model, feats, x, [layer_name])[0].detach().cpu()
    idx = torch.tensor(stats["idx"])
    fm = fm[:, idx, :, :]
    b, d, h, w = fm.shape
    feats_np = fm.permute(0, 2, 3, 1).reshape(b, h * w, d).numpy()
    mu = stats["mu"]
    cov_inv = stats["cov_inv"]
    heat = np.zeros((b, h * w), dtype=np.float32)
    for i in range(h * w):
        diff = feats_np[:, i, :] - mu[i]
        tmp = diff @ cov_inv[i]
        heat[:, i] = np.sum(tmp * diff, axis=1)
    heat = heat.reshape(b, h, w)
    heat_t = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(heat_t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    heat_up = heat_up.squeeze(1).numpy()
    flat = heat_up.reshape(b, -1)
    scores = np.mean(np.sort(flat, axis=1)[:, -PATCH_SCORE_TOPK:], axis=1)
    return scores, heat_up

# Define the small convolutional autoencoder baseline.
class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU())
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1), nn.Sigmoid())
    def forward(self, x):
        return self.dec(self.enc(x))

# Train the autoencoder baseline on normal images only.
def train_ae(model, train_loader, epochs, lr):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    epoch_losses = []
    t0_all = time.time()
    # Train the autoencoder for a small number of epochs on normal images.
    for ep in range(1, epochs + 1):
        losses = []
        for x, _, _, _ in train_loader:
            x = x.to(DEVICE)
            out = model(x)
            loss = F.mse_loss(out, x)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        ep_loss = float(np.mean(losses))
        epoch_losses.append(ep_loss)
        print(f"AE ep {ep:02d} | loss={ep_loss:.5f}")
    return {"epoch_losses": epoch_losses, "fit_sec": time.time() - t0_all}

@torch.no_grad()
def ae_scores(model, x):
    model.eval()
    x = x.to(DEVICE)
    out = model(x)
    err = (out - x).pow(2).mean(dim=1)
    heat = err.detach().cpu().numpy()
    flat = heat.reshape(heat.shape[0], -1)
    scores = np.mean(np.sort(flat, axis=1)[:, -PATCH_SCORE_TOPK:], axis=1)
    return scores, heat

# Evaluate one method on the test loader and return the main metrics.
def eval_method(test_loader, score_fn):
    y_true, y_score, masks, heats = [], [], [], []
    t0, n_img = time.time(), 0
    for x, y, m, _ in test_loader:
        scores, heat = score_fn(x)
        y_true.extend(y.numpy().tolist())
        y_score.extend(scores.tolist())
        m_np = m.squeeze(1).numpy()
        for i in range(m_np.shape[0]):
            masks.append(m_np[i])
            heats.append(heat[i])
        n_img += x.shape[0]
    sec = time.time() - t0
    return {
        "img_roc_auc": safe_roc_auc(y_true, y_score),
        "img_pr_auc": safe_pr_auc(y_true, y_score),
        "px_roc_auc": pixel_roc_auc(masks, heats),
        "sec_per_img": sec / max(n_img, 1),
        "n_eval_images": int(n_img)}

# Pick a few normal and anomalous items for the qualitative figure.
def select_qual_items(test_items, n_good=3, n_bad=3):
    good_items = [x for x in test_items if int(x["label"]) == 0][:n_good]
    bad_items = [x for x in test_items if int(x["label"]) == 1][:n_bad]
    return good_items + bad_items

In [ ]:
#------------------------------------------------------------------------------
# 7.2 Run baselines per category
#------------------------------------------------------------------------------
# Save the baseline settings once, then run the same three baselines for each category.
baseline_settings = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "mvtec_dir": str(MVTEC_DIR),
    "categories": CATEGORIES,
    "image_size": IMG_SIZE,
    "val_frac": VAL_FRAC,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_test": BATCH_SIZE_TEST,
    "backbone_name": BACKBONE_NAME,
    "patch_layer_names": PATCH_LAYER_NAMES,
    "padim_layer_name": PADIM_LAYER_NAME,
    "patchcore": {
        "coreset_ratio": CORESET_RATIO,
        "max_train_patches": MAX_TRAIN_PATCHES,
        "score_topk_pixels": PATCH_SCORE_TOPK},
    "padim": {
        "dim": PADIM_DIM,
        "eps": PADIM_EPS,
        "score_topk_pixels": PATCH_SCORE_TOPK},
    "autoencoder": {"epochs": AE_EPOCHS, "lr": AE_LR}}
write_json_overwrite(baseline_settings, BASELINE_SETTINGS_PATH)

backbone = get_resnet_imagenet().to(DEVICE)
feature_names = sorted(set(PATCH_LAYER_NAMES + [PADIM_LAYER_NAME]))
feats, _ = make_feature_hooks(backbone, feature_names)

rows_baselines = []
QUAL_CACHE = None

# Run the three baseline methods category by category.
for cat in CATEGORIES:
    print("\n" + "-" * 90)
    print("CATEGORY:", cat)
    print("-" * 90)

    train_items = splits[cat]["train_good"]
    val_items = splits[cat]["val_good"]
    test_items = splits[cat]["test"]

    n_test_total = len(test_items)
    n_test_anomaly = sum(int(x["label"]) == 1 for x in test_items)

    train_loader_im, _, test_loader_im = make_split_loaders(cat, input_kind="imagenet")
    train_loader_ae, _, test_loader_ae = make_split_loaders(cat, input_kind="ae")

    # ImageNet PatchCore
    t0 = time.time()
    bank = build_patch_bank(backbone, feats, train_loader_im, PATCH_LAYER_NAMES)
    index = faiss_index_l2(bank)
    patchcore_fit_sec = time.time() - t0

    # Score a batch with the fitted PatchCore memory bank.
    def score_fn_pc(x):
        return patchcore_scores(backbone, feats, x, PATCH_LAYER_NAMES, index)

    res_pc = eval_method(test_loader_im, score_fn_pc)
    row_pc = {
        "category": cat, "method": "imagenet_patchcore", "run_mode": RUN_MODE,
        "representation_source": "imagenet", "anomaly_head": "patchcore", "aug_strength": "none",
        "backbone_name": BACKBONE_NAME, "layers": ",".join(PATCH_LAYER_NAMES), "img_size": IMG_SIZE,
        "coreset_ratio": CORESET_RATIO,
        "n_train_good": len(train_items), "n_val_good": len(val_items), "n_test_total": n_test_total, "n_test_anomaly": n_test_anomaly,
        "fit_sec": patchcore_fit_sec, "feature_dim": int(bank.shape[1]), "memory_bank_n": int(bank.shape[0]),
        "memory_bank_mb": float(bank.nbytes / (1024 ** 2)), "checkpoint_size_mb": np.nan, "peak_gpu_mem_mb": np.nan,
        **res_pc}
    rows_baselines.append(row_pc)
    print("PatchCore:", {k: row_pc[k] for k in ["img_roc_auc","img_pr_auc","px_roc_auc","fit_sec","sec_per_img"]})

    # ImageNet PaDiM
    t0 = time.time()
    stats = padim_fit(backbone, feats, train_loader_im, layer_name=PADIM_LAYER_NAME)
    padim_fit_sec = time.time() - t0
    padim_mb = (stats["mu"].nbytes + stats["cov_inv"].nbytes) / (1024 ** 2)

    # Score a batch with the fitted PaDiM statistics.
    def score_fn_pd(x):
        return padim_scores(backbone, feats, x, layer_name=PADIM_LAYER_NAME, stats=stats)

    res_pd = eval_method(test_loader_im, score_fn_pd)
    row_pd = {
        "category": cat, "method": "imagenet_padim", "run_mode": RUN_MODE,
        "representation_source": "imagenet", "anomaly_head": "padim", "aug_strength": "none",
        "backbone_name": BACKBONE_NAME, "layers": PADIM_LAYER_NAME, "img_size": IMG_SIZE,
        "coreset_ratio": np.nan,
        "n_train_good": len(train_items), "n_val_good": len(val_items), "n_test_total": n_test_total, "n_test_anomaly": n_test_anomaly,
        "fit_sec": padim_fit_sec, "feature_dim": int(stats["D"]), "memory_bank_n": int(stats["H"] * stats["W"]),
        "memory_bank_mb": float(padim_mb), "checkpoint_size_mb": np.nan, "peak_gpu_mem_mb": np.nan,
        **res_pd}
    rows_baselines.append(row_pd)
    print("PaDiM:", {k: row_pd[k] for k in ["img_roc_auc","img_pr_auc","px_roc_auc","fit_sec","sec_per_img"]})

    # Autoencoder
    ae = SmallAE()
    ae_hist = train_ae(ae, train_loader_ae, epochs=AE_EPOCHS, lr=AE_LR)
    ckpt_path = CHECKPOINTS_DIR / f"ae_{cat}.pt"
    torch.save(ae.state_dict(), ckpt_path)

    # Score a batch with the trained autoencoder baseline.
    def score_fn_ae(x):
        return ae_scores(ae, x)

    res_ae = eval_method(test_loader_ae, score_fn_ae)
    row_ae = {
        "category": cat, "method": "autoencoder", "run_mode": RUN_MODE,
        "representation_source": "none", "anomaly_head": "autoencoder", "aug_strength": "none",
        "backbone_name": "small_conv_ae", "layers": "reconstruction_error", "img_size": IMG_SIZE,
        "coreset_ratio": np.nan,
        "n_train_good": len(train_items), "n_val_good": len(val_items), "n_test_total": n_test_total, "n_test_anomaly": n_test_anomaly,
        "fit_sec": float(ae_hist["fit_sec"]), "feature_dim": np.nan, "memory_bank_n": np.nan, "memory_bank_mb": np.nan,
        "checkpoint_size_mb": float(file_size_mb(ckpt_path)), "peak_gpu_mem_mb": np.nan,
        **res_ae}
    rows_baselines.append(row_ae)
    print("AE:", {k: row_ae[k] for k in ["img_roc_auc","img_pr_auc","px_roc_auc","fit_sec","sec_per_img"]})

    # Keep one category aside for qualitative heatmap examples.
    if cat == QUAL_CATEGORY:
        qual_items = select_qual_items(test_items, n_good=3, n_bad=3)
        qual_loader_im = make_loader(qual_items, mode="test", img_tfm=tfm_imagenet, batch_size=len(qual_items), shuffle=False)
        qual_loader_ae = make_loader(qual_items, mode="test", img_tfm=tfm_ae, batch_size=len(qual_items), shuffle=False)

        x_im, y_q, m_q, paths_q = next(iter(qual_loader_im))
        x_ae, _, _, _ = next(iter(qual_loader_ae))

        _, heat_pc = patchcore_scores(backbone, feats, x_im, PATCH_LAYER_NAMES, index)
        _, heat_pd = padim_scores(backbone, feats, x_im, PADIM_LAYER_NAME, stats)
        _, heat_ae = ae_scores(ae, x_ae)

        QUAL_CACHE = {"category": cat, "x_ae": x_ae, "labels": y_q.numpy(), "masks": m_q.squeeze(1).numpy(),
                      "paths": list(paths_q), "heat_pc": heat_pc, "heat_pd": heat_pd, "heat_ae": heat_ae}

    del bank, index, stats, ae
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# Combine all baseline rows and then add a MEAN row per method.
df_baselines = pd.DataFrame(rows_baselines)
metric_cols = ["img_roc_auc", "img_pr_auc", "px_roc_auc","fit_sec", "sec_per_img", "feature_dim", "memory_bank_n", "memory_bank_mb", "checkpoint_size_mb"]
mean_df_base = df_baselines.groupby(["method", "representation_source", "anomaly_head", "aug_strength"],as_index=False)[metric_cols].mean(numeric_only=True)
mean_df_base["category"] = "MEAN"
mean_df_base["run_mode"] = RUN_MODE
mean_df_base["backbone_name"] = ""
mean_df_base["layers"] = PATCH_LAYER_NAME_MAIN
mean_df_base["img_size"] = IMG_SIZE
mean_df_base["coreset_ratio"] = mean_df_base["anomaly_head"].map({"patchcore": CORESET_RATIO, "padim": np.nan, "autoencoder": np.nan})
mean_df_base["n_train_good"] = np.nan
mean_df_base["n_val_good"] = np.nan
mean_df_base["n_test_total"] = np.nan
mean_df_base["n_test_anomaly"] = np.nan
mean_df_base["peak_gpu_mem_mb"] = np.nan
mean_df_base["n_eval_images"] = np.nan

ordered_cols = ["category", "method", "run_mode", "representation_source", "anomaly_head", "aug_strength","backbone_name", "layers", "img_size", "coreset_ratio",
                "n_train_good", "n_val_good", "n_test_total", "n_test_anomaly","img_roc_auc", "img_pr_auc", "px_roc_auc","fit_sec", "sec_per_img", "n_eval_images",
                "feature_dim", "memory_bank_n", "memory_bank_mb", "checkpoint_size_mb", "peak_gpu_mem_mb"]
table_baselines = pd.concat([df_baselines[ordered_cols], mean_df_base[ordered_cols]], ignore_index=True)
table_baselines.to_csv(TABLE_BASELINES_PATH, index=False)

print_banner("Baseline summary table")
display(table_baselines.tail(10))
print("Saved:", TABLE_BASELINES_PATH)

In [ ]:
#------------------------------------------------------------------------------
# 7.3 Baseline figures
#------------------------------------------------------------------------------
# Save one compact summary figure plus a qualitative heatmap figure.
METHOD_ORDER = ["imagenet_patchcore", "imagenet_padim", "autoencoder"]
mean_plot_df = (df_baselines.groupby("method")[["img_roc_auc", "img_pr_auc", "px_roc_auc", "sec_per_img"]].mean(numeric_only=True).reindex(METHOD_ORDER).reset_index())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_specs = [("img_roc_auc", "Mean image ROC-AUC"),
    ("img_pr_auc", "Mean image PR-AUC"),
    ("px_roc_auc", "Mean pixel ROC-AUC"),
    ("sec_per_img", "Mean inference sec / image")]
for ax, (metric, title) in zip(axes.ravel(), plot_specs):
    ax.bar(mean_plot_df["method"], mean_plot_df[metric])
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
    if "auc" in metric:
        ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.savefig(FIG_BASELINES_PATH, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", FIG_BASELINES_PATH)

x_ae = QUAL_CACHE["x_ae"]
labels = QUAL_CACHE["labels"]
masks = QUAL_CACHE["masks"]
paths = QUAL_CACHE["paths"]
heat_pc = QUAL_CACHE["heat_pc"]
heat_pd = QUAL_CACHE["heat_pd"]
heat_ae = QUAL_CACHE["heat_ae"]

# Plot one row per saved qualitative example.
b = x_ae.shape[0]
plt.figure(figsize=(15, 3.0 * b))
for i in range(b):
    row_name = f"{'anomaly' if labels[i] == 1 else 'good'} | {Path(paths[i]).parent.name}"
    ax = plt.subplot(b, 5, i * 5 + 1)
    overlay(ax, x_ae[i], None, title="Image")
    ax.set_ylabel(row_name, fontsize=9)

    ax = plt.subplot(b, 5, i * 5 + 2)
    show_mask(ax, masks[i], title="GT mask")

    ax = plt.subplot(b, 5, i * 5 + 3)
    overlay(ax, x_ae[i], heat_pc[i], title="PatchCore")

    ax = plt.subplot(b, 5, i * 5 + 4)
    overlay(ax, x_ae[i], heat_pd[i], title="PaDiM")

    ax = plt.subplot(b, 5, i * 5 + 5)
    overlay(ax, x_ae[i], heat_ae[i], title="AE")

plt.tight_layout()
plt.savefig(FIG_HEATMAPS_BASELINES_PATH, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", FIG_HEATMAPS_BASELINES_PATH)

# 8.0 SimCLR pretraining

## Purpose
- train a self-supervised encoder using only defect-free training images
- test whether in-domain representation learning can provide useful features for downstream anomaly detection
- compare different augmentation strengths so that the effect of the SimCLR pretraining setup can be studied rather than assumed
- save checkpoints, loss tables, and training curves so that the downstream evaluation is traceable and reproducible
- provide a clear bridge between the representation-learning stage and the final anomaly detection comparison

## Process
1. collect the normal training images across the active categories.
2. define the SimCLR data pipeline, including the paired augmentations used to create positive views.
3. build the encoder, projection head, and contrastive loss used during pretraining.
4. train the requested augmentation settings and save the resulting checkpoints and run summaries.
5. reuse the saved encoder checkpoints later in the final comparison and ablation study.

In [ ]:
#------------------------------------------------------------------------------
# 8.1 SimCLR dataset, augmentations, and model
#------------------------------------------------------------------------------
ssl_rows, ssl_paths = [], []
# Pool together all defect-free training images used for SSL pretraining.
for cat in CATEGORIES:
    cat_paths = splits[cat]["train_good"]
    ssl_paths.extend(cat_paths)
    ssl_rows.append({"category": cat, "n_train_good": len(cat_paths)})

df_ssl_coverage = pd.DataFrame(ssl_rows).sort_values("category").reset_index(drop=True)
print_banner("SSL training coverage")
display(df_ssl_coverage)
print("Total SSL train_good images:", len(ssl_paths))

# Apply Gaussian blur as one of the SimCLR augmentations.
class GaussianBlur:
    def __init__(self, p=0.5, radius_min=0.1, radius_max=2.0):
        self.p = p
        self.radius_min = radius_min
        self.radius_max = radius_max
    def __call__(self, img):
        if random.random() > self.p:
            return img
        r = float(random.uniform(self.radius_min, self.radius_max))
        return img.filter(ImageFilter.GaussianBlur(radius=r))

# Build the mild or strong augmentation policy used in the SSL study.
def build_simclr_transform(img_size, strength="mild"):
    if strength == "mild":
        cj = transforms.ColorJitter(0.20, 0.20, 0.10, 0.05)
        blur = GaussianBlur(p=0.20, radius_min=0.1, radius_max=1.0)
        crop_scale = (0.60, 1.00)
        gray_p = 0.05
        flip_p = 0.50
    elif strength == "strong":
        cj = transforms.ColorJitter(0.40, 0.40, 0.20, 0.10)
        blur = GaussianBlur(p=0.50, radius_min=0.1, radius_max=2.0)
        crop_scale = (0.40, 1.00)
        gray_p = 0.10
        flip_p = 0.50
    else:
        raise ValueError(f"Unknown strength: {strength}")

    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=crop_scale),
        transforms.RandomHorizontalFlip(p=flip_p),
        transforms.RandomApply([cj], p=0.8),
        transforms.RandomGrayscale(p=gray_p),
        blur,
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),])

# Return two augmented views of each defect-free training image.
class SimclrDataset(Dataset):
    def __init__(self, img_paths, transform):
        self.img_paths = list(img_paths)
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, i):
        img = Image.open(self.img_paths[i]).convert("RGB")
        x1 = self.transform(img)
        x2 = self.transform(img)
        return x1, x2

# Create the SSL DataLoader from the pooled normal training images.
def make_simclr_loader(img_paths, aug_strength):
    ds = SimclrDataset(img_paths, build_simclr_transform(IMG_SIZE, strength=aug_strength))
    loader_kwargs = dict(
        batch_size=SIMCLR_BATCH_SIZE_SINGLE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        drop_last=True)
    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(ds, **loader_kwargs)

# Build the ResNet encoder before the projection head is attached.
def get_resnet18_encoder():
    m = torchvision.models.resnet18(weights=None)
    feat_dim = m.fc.in_features
    m.fc = nn.Identity()
    return m, feat_dim

# Map encoder features into the contrastive projection space.
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),)
    def forward(self, x):
        return self.net(x)

# Combine the encoder and projector into one SimCLR model.
class SimclrModel(nn.Module):
    def __init__(self, encoder, feat_dim, proj_dim=128):
        super().__init__()
        self.encoder = encoder
        self.projector = ProjectionHead(feat_dim, hidden_dim=512, out_dim=proj_dim)
    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)

# Compute the standard SimCLR contrastive loss.
def nt_xent_loss(z1, z2, temperature=0.2):
    b = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.t()) / temperature
    diag = torch.eye(2 * b, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(diag, torch.finfo(sim.dtype).min)
    targets = torch.arange(2 * b, device=z.device)
    targets = (targets + b) % (2 * b)
    return F.cross_entropy(sim.float(), targets)

In [ ]:
#------------------------------------------------------------------------------
# 8.2 SimCLR training helpers
#------------------------------------------------------------------------------
# Save the loss curve so the SSL training behaviour is easy to report later.
def plot_and_save_loss(df_train, fig_path, aug_strength):
    plt.figure(figsize=(7, 4))
    plt.plot(df_train["epoch"], df_train["loss"], marker="o")
    plt.title(f"SimCLR Training Loss ({aug_strength.capitalize()} Augmentation)")
    plt.xlabel("Epoch")
    plt.ylabel("NT-Xent Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(fig_path, dpi=220, bbox_inches="tight")
    plt.show()

# Run one SimCLR setting in a normal single Python process.
def run_one_simclr_train_single(aug_strength):
    print_banner(f"SimCLR run (single process): {aug_strength}")

    train_loader = make_simclr_loader(ssl_paths, aug_strength=aug_strength)
    encoder, feat_dim = get_resnet18_encoder()
    model = SimclrModel(encoder, feat_dim, proj_dim=SIMCLR_PROJ_DIM).to(DEVICE)
    if SIMCLR_MULTI_GPU_POLICY == "dp_fallback" and DEVICE == "cuda" and GPU_COUNT > 1:
        model = nn.DataParallel(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=SIMCLR_LR, weight_decay=SIMCLR_WEIGHT_DECAY)
    use_amp = (DEVICE == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    encoder_path = CHECKPOINTS_DIR / f"simclr_{aug_strength}_encoder.pt"
    full_path = CHECKPOINTS_DIR / f"simclr_{aug_strength}_full.pt"
    table_path = RESULTS_DIR / f"table_simclr_train_{aug_strength}.csv"
    fig_path = RESULTS_DIR / f"fig_simclr_loss_{aug_strength}.png"
    meta_path = ARTIFACTS_DIR / f"simclr_{aug_strength}_meta.json"

    # Keep one row per epoch for the loss table and figure.
    rows = []
    best_loss = np.inf
    best_epoch = None

    reset_peak_gpu_memory()
    train_t0 = time.time()

    # Train over the augmented view pairs for the chosen SSL setting.
    for ep in range(1, SIMCLR_EPOCHS + 1):
        model.train()
        ep_t0 = time.time()
        losses = []

        pbar = tqdm(train_loader, desc=f"{aug_strength} | epoch {ep:02d}/{SIMCLR_EPOCHS}", leave=False)
        for x1, x2 in pbar:
            x1 = x1.to(DEVICE, non_blocking=True)
            x2 = x2.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                x = torch.cat([x1, x2], dim=0)
                z = model(x)
                z1, z2 = torch.chunk(z, 2, dim=0)
                loss = nt_xent_loss(z1, z2, temperature=SIMCLR_TEMPERATURE)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            losses.append(loss.item())
            pbar.set_postfix(loss=float(np.mean(losses)))

        ep_loss = float(np.mean(losses))
        ep_sec = float(time.time() - ep_t0)
        rows.append({"epoch": ep, "loss": ep_loss, "sec": ep_sec, "aug_strength": aug_strength, "run_mode": RUN_MODE})

        # Save the best encoder because that is what the downstream methods use later.
        if ep_loss < best_loss:
            best_loss = ep_loss
            best_epoch = ep
            save_model = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(save_model.encoder.state_dict(), encoder_path)
            if SIMCLR_SAVE_FULL_MODEL_BACKUP:
                torch.save(save_model.state_dict(), full_path)

        print(f"Epoch {ep:02d} | loss={ep_loss:.4f} | sec={ep_sec:.1f}")

    total_sec = float(time.time() - train_t0)
    peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    df_train = pd.DataFrame(rows)
    df_train.to_csv(table_path, index=False)
    plot_and_save_loss(df_train, fig_path, aug_strength=aug_strength)

    meta = {
        "timestamp": now_stamp(),
        "run_mode": RUN_MODE,
        "aug_strength": aug_strength,
        "categories_ssl": CATEGORIES,
        "n_categories_ssl": len(CATEGORIES),
        "n_train_images_ssl": len(ssl_paths),
        "image_size": IMG_SIZE,
        "batch_size_per_process": SIMCLR_BATCH_SIZE_SINGLE,
        "epochs": SIMCLR_EPOCHS,
        "learning_rate": SIMCLR_LR,
        "weight_decay": SIMCLR_WEIGHT_DECAY,
        "temperature": SIMCLR_TEMPERATURE,
        "proj_dim": SIMCLR_PROJ_DIM,
        "backbone_name": "resnet18",
        "amp_enabled": use_amp,
        "multi_gpu_mode": "dataparallel" if isinstance(model, nn.DataParallel) else "single",
        "total_train_sec": total_sec,
        "mean_epoch_sec": float(df_train["sec"].mean()),
        "best_loss": float(best_loss),
        "best_epoch": int(best_epoch),
        "peak_gpu_mem_mb": peak_gpu_mem_mb,
        "encoder_checkpoint_path": str(encoder_path),
        "encoder_checkpoint_size_mb": float(file_size_mb(encoder_path)),
        "full_model_checkpoint_path": str(full_path) if SIMCLR_SAVE_FULL_MODEL_BACKUP else None,
        "full_model_checkpoint_size_mb": float(file_size_mb(full_path)) if SIMCLR_SAVE_FULL_MODEL_BACKUP else np.nan,
        "loss_table_path": str(table_path),
        "loss_curve_path": str(fig_path)}
    write_json_overwrite(meta, meta_path)

    return {
        "run_mode": RUN_MODE,
        "aug_strength": aug_strength,
        "n_categories_ssl": len(CATEGORIES),
        "n_train_images_ssl": len(ssl_paths),
        "image_size": IMG_SIZE,
        "batch_size": SIMCLR_BATCH_SIZE_SINGLE,
        "epochs": SIMCLR_EPOCHS,
        "learning_rate": SIMCLR_LR,
        "weight_decay": SIMCLR_WEIGHT_DECAY,
        "temperature": SIMCLR_TEMPERATURE,
        "best_loss": float(best_loss),
        "best_epoch": int(best_epoch),
        "total_train_sec": total_sec,
        "mean_epoch_sec": float(df_train["sec"].mean()),
        "peak_gpu_mem_mb": peak_gpu_mem_mb,
        "encoder_checkpoint_size_mb": float(file_size_mb(encoder_path)),
        "encoder_checkpoint_path": str(encoder_path),
        "loss_table_path": str(table_path),
        "loss_curve_path": str(fig_path),
        "multi_gpu_mode": meta["multi_gpu_mode"]}

# Run one SimCLR setting with torchrun so each GPU gets its own process which requires a script
def run_one_simclr_train_ddp(aug_strength):
    print_banner(f"SimCLR run (DDP): {aug_strength}")

    manifest_path = ARTIFACTS_DIR / f"simclr_{aug_strength}_manifest.json"
    table_path = RESULTS_DIR / f"table_simclr_train_{aug_strength}.csv"
    meta_path = ARTIFACTS_DIR / f"simclr_{aug_strength}_meta.json"
    encoder_path = CHECKPOINTS_DIR / f"simclr_{aug_strength}_encoder.pt"
    full_path = CHECKPOINTS_DIR / f"simclr_{aug_strength}_full.pt"

    # Store the run settings in JSON so the helper script stays short.
    manifest = {
        "ssl_paths": ssl_paths,
        "aug_strength": aug_strength,
        "img_size": IMG_SIZE,
        "epochs": SIMCLR_EPOCHS,
        "batch_size": SIMCLR_BATCH_SIZE_SINGLE,
        "lr": SIMCLR_LR,
        "weight_decay": SIMCLR_WEIGHT_DECAY,
        "temperature": SIMCLR_TEMPERATURE,
        "proj_dim": SIMCLR_PROJ_DIM,
        "num_workers": 2 if NUM_WORKERS > 0 else 0,
        "seed": SEED,
        "table_path": str(table_path),
        "meta_path": str(meta_path),
        "encoder_path": str(encoder_path),
        "full_path": str(full_path),
        "save_full": bool(SIMCLR_SAVE_FULL_MODEL_BACKUP),
        "run_mode": RUN_MODE,
        "categories": CATEGORIES}
    write_json_overwrite(manifest, manifest_path)

#DDP script needed so each GPU can be run.
    ddp_script_path = WORK_ROOT / "_simclr_ddp_train.py"
    ddp_script = '''
import os, sys, json, time, random
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
import torchvision
from torchvision import transforms

# Read the manifest written by the notebook.
def read_json(path):
    with open(path, "r") as f:
        return json.load(f)

# Save the DDP training metadata back to disk.
def write_json_overwrite(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# Set the random seeds inside each DDP worker process.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# Apply blur as one of the SimCLR augmentations.
class GaussianBlur:
    def __init__(self, p=0.5, radius_min=0.1, radius_max=2.0):
        self.p = p
        self.radius_min = radius_min
        self.radius_max = radius_max

    def __call__(self, img):
        if random.random() > self.p:
            return img
        r = float(random.uniform(self.radius_min, self.radius_max))
        return img.filter(ImageFilter.GaussianBlur(radius=r))

# Build the mild or strong augmentation policy.
def build_simclr_transform(img_size, strength="mild"):
    if strength == "mild":
        cj = transforms.ColorJitter(0.20, 0.20, 0.10, 0.05)
        blur = GaussianBlur(p=0.20, radius_min=0.1, radius_max=1.0)
        crop_scale = (0.60, 1.00)
        gray_p = 0.05
        flip_p = 0.50
    else:
        cj = transforms.ColorJitter(0.40, 0.40, 0.20, 0.10)
        blur = GaussianBlur(p=0.50, radius_min=0.1, radius_max=2.0)
        crop_scale = (0.40, 1.00)
        gray_p = 0.10
        flip_p = 0.50

    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=crop_scale),
        transforms.RandomHorizontalFlip(p=flip_p),
        transforms.RandomApply([cj], p=0.8),
        transforms.RandomGrayscale(p=gray_p),
        blur,
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

# Return two augmented views of the same image.
class SimclrDataset(Dataset):
    def __init__(self, img_paths, transform):
        self.img_paths = list(img_paths)
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, i):
        img = Image.open(self.img_paths[i]).convert("RGB")
        return self.transform(img), self.transform(img)

# Build the ResNet encoder used before the projection head.
def get_resnet18_encoder():
    m = torchvision.models.resnet18(weights=None)
    feat_dim = m.fc.in_features
    m.fc = nn.Identity()
    return m, feat_dim

# Map encoder features into the contrastive projection space.
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),)

    def forward(self, x):
        return self.net(x)

# Wrap the encoder and projection head into one SimCLR model.
class SimclrModel(nn.Module):
    def __init__(self, encoder, feat_dim, proj_dim=128):
        super().__init__()
        self.encoder = encoder
        self.projector = ProjectionHead(feat_dim, hidden_dim=512, out_dim=proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)

# Compute the standard SimCLR contrastive loss.
def nt_xent_loss(z1, z2, temperature=0.2):
    b = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.t()) / temperature
    diag = torch.eye(2 * b, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(diag, torch.finfo(sim.dtype).min)
    targets = torch.arange(2 * b, device=z.device)
    targets = (targets + b) % (2 * b)
    return F.cross_entropy(sim.float(), targets)

# Run the full DDP training loop and save the best encoder on rank 0.
def main(manifest_path):
    cfg = read_json(manifest_path)
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")
    set_seed(cfg["seed"] + rank)

    ds = SimclrDataset(cfg["ssl_paths"], build_simclr_transform(cfg["img_size"], cfg["aug_strength"]))
    sampler = DistributedSampler(ds, shuffle=True, drop_last=True)
    loader = DataLoader(
        ds,
        batch_size=cfg["batch_size"],
        sampler=sampler,
        num_workers=cfg["num_workers"],
        pin_memory=True,
        drop_last=True,
        persistent_workers=(cfg["num_workers"] > 0))

    encoder, feat_dim = get_resnet18_encoder()
    model = SimclrModel(encoder, feat_dim, proj_dim=cfg["proj_dim"]).cuda(local_rank)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    rows = []
    best_loss = float("inf")
    best_epoch = None
    t0_all = time.time()
    torch.cuda.reset_peak_memory_stats(local_rank)

    # Each rank trains on its own shard and rank 0 keeps the summary rows.
    for ep in range(1, cfg["epochs"] + 1):
        sampler.set_epoch(ep)
        model.train()
        losses = []
        t0 = time.time()

        for x1, x2 in loader:
            x1 = x1.cuda(local_rank, non_blocking=True)
            x2 = x2.cuda(local_rank, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
                x = torch.cat([x1, x2], dim=0)
                z = model(x)
                z1, z2 = torch.chunk(z, 2, dim=0)
                loss = nt_xent_loss(z1, z2, temperature=cfg["temperature"])

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())

        mean_loss = torch.tensor([float(np.mean(losses))], device=local_rank)
        dist.all_reduce(mean_loss, op=dist.ReduceOp.SUM)
        mean_loss = (mean_loss / world_size).item()
        ep_sec = time.time() - t0

        if rank == 0:
            rows.append({
                "epoch": ep,
                "loss": float(mean_loss),
                "sec": float(ep_sec),
                "aug_strength": cfg["aug_strength"],
                "run_mode": cfg["run_mode"]
            })

            if mean_loss < best_loss:
                best_loss = mean_loss
                best_epoch = ep
                torch.save(model.module.encoder.state_dict(), cfg["encoder_path"])
                if cfg["save_full"]:
                    torch.save(model.module.state_dict(), cfg["full_path"])

    if rank == 0:
        df = pd.DataFrame(rows)
        df.to_csv(cfg["table_path"], index=False)

        meta = {
            "run_mode": cfg["run_mode"],
            "aug_strength": cfg["aug_strength"],
            "categories_ssl": cfg["categories"],
            "n_categories_ssl": len(cfg["categories"]),
            "n_train_images_ssl": len(cfg["ssl_paths"]),
            "image_size": cfg["img_size"],
            "batch_size_per_process": cfg["batch_size"],
            "effective_batch_size": cfg["batch_size"] * world_size,
            "epochs": cfg["epochs"],
            "learning_rate": cfg["lr"],
            "weight_decay": cfg["weight_decay"],
            "temperature": cfg["temperature"],
            "proj_dim": cfg["proj_dim"],
            "backbone_name": "resnet18",
            "amp_enabled": True,
            "multi_gpu_mode": f"ddp_{world_size}gpu",
            "total_train_sec": float(time.time() - t0_all),
            "mean_epoch_sec": float(df["sec"].mean()),
            "best_loss": float(best_loss),
            "best_epoch": int(best_epoch),
            "peak_gpu_mem_mb": float(torch.cuda.max_memory_allocated(local_rank) / (1024 ** 2)),
            "encoder_checkpoint_path": cfg["encoder_path"],
            "loss_table_path": cfg["table_path"]}
        write_json_overwrite(meta, cfg["meta_path"])

    dist.barrier()
    dist.destroy_process_group()

if __name__ == "__main__":
    main(sys.argv[1])
'''
    ddp_script_path.write_text(ddp_script)

    torchrun_path = shutil.which("torchrun")
    cmd = [torchrun_path, "--standalone", "--nproc_per_node", str(GPU_COUNT), str(ddp_script_path), str(manifest_path)]
    print("Launching:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    meta = read_json(meta_path)
    df_train = pd.read_csv(table_path)
    fig_path = RESULTS_DIR / f"fig_simclr_loss_{aug_strength}.png"
    plot_and_save_loss(df_train, fig_path, aug_strength=aug_strength)

    meta["loss_curve_path"] = str(fig_path)
    write_json_overwrite(meta, meta_path)

    return {
        "run_mode": RUN_MODE,
        "aug_strength": aug_strength,
        "n_categories_ssl": meta["n_categories_ssl"],
        "n_train_images_ssl": meta["n_train_images_ssl"],
        "image_size": meta["image_size"],
        "batch_size": meta["batch_size_per_process"],
        "epochs": meta["epochs"],
        "learning_rate": meta["learning_rate"],
        "weight_decay": meta["weight_decay"],
        "temperature": meta["temperature"],
        "best_loss": meta["best_loss"],
        "best_epoch": meta["best_epoch"],
        "total_train_sec": meta["total_train_sec"],
        "mean_epoch_sec": meta["mean_epoch_sec"],
        "peak_gpu_mem_mb": meta["peak_gpu_mem_mb"],
        "encoder_checkpoint_size_mb": float(file_size_mb(encoder_path)),
        "encoder_checkpoint_path": str(encoder_path),
        "loss_table_path": str(table_path),
        "loss_curve_path": str(fig_path),
        "multi_gpu_mode": meta["multi_gpu_mode"]}

# Choose the training path based on the GPU setup and selected policy.
def run_one_simclr_train(aug_strength):
    use_ddp = (SIMCLR_MULTI_GPU_POLICY == "ddp_if_available" and DEVICE == "cuda" and GPU_COUNT > 1)
    if use_ddp:
        return run_one_simclr_train_ddp(aug_strength)
    return run_one_simclr_train_single(aug_strength)

In [ ]:
#------------------------------------------------------------------------------
# 8.3 Run requested SimCLR augmentation settings
#------------------------------------------------------------------------------
summary_rows = []
for aug_strength in SIMCLR_AUG_STRENGTHS:
    summary_rows.append(run_one_simclr_train(aug_strength))

df_simclr_summary = pd.DataFrame(summary_rows).sort_values("aug_strength").reset_index(drop=True)
df_simclr_summary.to_csv(TABLE_SIMCLR_SUMMARY_PATH, index=False)

runs_payload = {"timestamp": now_stamp(),
                "run_mode": RUN_MODE,
                "categories_ssl": CATEGORIES,
                "runs": df_simclr_summary.to_dict(orient="records")}
write_json_overwrite(runs_payload, SIMCLR_RUNS_JSON_PATH)

print_banner("SimCLR summary")
display(df_simclr_summary)
print("Saved:", TABLE_SIMCLR_SUMMARY_PATH)
print("Saved:", SIMCLR_RUNS_JSON_PATH)

SSL_CKPT_MAP = {tag: CHECKPOINTS_DIR / f"simclr_{tag}_encoder.pt" for tag in ["mild", "strong"]}
AVAILABLE_SSL_TAGS = [tag for tag, p in SSL_CKPT_MAP.items() if p.exists()]
print("Available SSL checkpoints:", AVAILABLE_SSL_TAGS)

# 9.0 Main comparison, thresholds, ablations, and failures

## Purpose
- run the main final comparison across all candidate methods using matched downstream settings
- compare methods fairly rather than mixing different evaluation rules for different models
- study threshold behaviour so the results are useful not only from an academic metric perspective but also from a practical deployment perspective
- run ablations to test how much performance depends on augmentation strength, embedding layer choice, and coreset size
- examine failure cases so the notebook does not only report average performance, but also shows where and how the methods make mistakes
- save the final summary tables that will be used in the report results section

## Process
1. rebuild each method in evaluation mode from the saved artefacts and shared settings.
2. run the matched main comparison across the active categories and save the summary results.
3. evaluate threshold policies on the most relevant methods to study the trade-off between recall and false alarms.
4. run targeted ablations for the key design choices.
5. perform failure analysis to inspect the kinds of errors made by the selected method.

In [ ]:
#------------------------------------------------------------------------------
# 9.1 Runtime builders for final evaluation
#------------------------------------------------------------------------------
# Rebuild each saved method in a consistent way before the final comparison stage.

# Load the SSL encoder weights that were saved after SimCLR training.
def load_ssl_encoder(ckpt_path):
    model = get_resnet18_ssl()
    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

# Load the saved autoencoder checkpoint for one category.
def load_ae_checkpoint(cat):
    ckpt_path = CHECKPOINTS_DIR / f"ae_{cat}.pt"
    model = SmallAE()
    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state, strict=True)
    model.eval()
    return model, ckpt_path

# Translate the method label back into the settings needed to rebuild it.
def parse_method_name(method_name):
    if method_name == "imagenet_patchcore":
        return {"representation_source": "imagenet", "anomaly_head": "patchcore", "aug_strength": "none"}
    if method_name == "imagenet_padim":
        return {"representation_source": "imagenet", "anomaly_head": "padim", "aug_strength": "none"}
    if method_name == "autoencoder":
        return {"representation_source": "none", "anomaly_head": "autoencoder", "aug_strength": "none"}

    _, aug_strength, anomaly_head = method_name.split("_")
    return {"representation_source": "simclr", "anomaly_head": anomaly_head, "aug_strength": aug_strength}

# Build the loaders, backbone, and score function for one feature-based method.
def build_feature_method_runtime(cat, representation_source, anomaly_head, layer_name, coreset_ratio, aug_strength=None):
    train_loader, val_loader, test_loader = make_split_loaders(cat, input_kind="imagenet")

    if representation_source == "imagenet":
        model = get_resnet_imagenet()
        ckpt_path = None
    else:
        ckpt_path = SSL_CKPT_MAP[aug_strength]
        model = load_ssl_encoder(ckpt_path)

    model = model.to(DEVICE)
    feats, _ = make_feature_hooks(model, [layer_name])

    reset_peak_gpu_memory()
    fit_t0 = time.time()

    if anomaly_head == "patchcore":
        bank = build_patch_bank(
            model, feats, train_loader, [layer_name],
            coreset_ratio=coreset_ratio, max_train_patches=MAX_TRAIN_PATCHES)
        index = faiss_index_l2(bank)

        # Score each image by the nearest normal patch distance.
        def score_fn(x):
            return patchcore_scores(model, feats, x, [layer_name], index)

        fit_stats = {
            "feature_dim": int(bank.shape[1]),
            "memory_bank_n": int(bank.shape[0]),
            "memory_bank_mb": float(bank.nbytes / (1024 ** 2))}

    else:
        stats = padim_fit(model, feats, train_loader, layer_name, PADIM_DIM, PADIM_EPS)

        # Score each image using the learned Gaussian patch statistics.
        def score_fn(x):
            return padim_scores(model, feats, x, layer_name, stats)

        fit_stats = {
            "feature_dim": int(stats["D"]),
            "memory_bank_n": int(stats["H"] * stats["W"]),
            "memory_bank_mb": float((stats["mu"].nbytes + stats["cov_inv"].nbytes) / (1024 ** 2))}

    fit_sec = float(time.time() - fit_t0)

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "score_fn": score_fn,
        "feature_dim": fit_stats["feature_dim"],
        "memory_bank_n": fit_stats["memory_bank_n"],
        "memory_bank_mb": fit_stats["memory_bank_mb"],
        "fit_sec": fit_sec,
        "peak_gpu_mem_mb": get_peak_gpu_memory_mb(),
        "checkpoint_size_mb": float(file_size_mb(ckpt_path)) if ckpt_path is not None else np.nan}

# Rebuild the saved autoencoder and return a score function in the same format.
def build_autoencoder_runtime(cat):
    _, val_loader, test_loader = make_split_loaders(cat, input_kind="ae")
    model, ckpt_path = load_ae_checkpoint(cat)
    model = model.to(DEVICE)

    # Use reconstruction error as the anomaly score.
    def score_fn(x):
        return ae_scores(model, x)

    return {
        "train_loader": None,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "score_fn": score_fn,
        "feature_dim": np.nan,
        "memory_bank_n": np.nan,
        "memory_bank_mb": np.nan,
        "fit_sec": np.nan,
        "peak_gpu_mem_mb": np.nan,
        "checkpoint_size_mb": float(file_size_mb(ckpt_path))}

# Route each method name to the matching runtime builder.
def build_method_runtime(cat, method_name):
    meta = parse_method_name(method_name)

    if method_name == "autoencoder":
        return {**meta, **build_autoencoder_runtime(cat)}

    return {**meta,
            **build_feature_method_runtime(
                cat=cat,
                representation_source=meta["representation_source"],
                anomaly_head=meta["anomaly_head"],
                layer_name=PATCH_LAYER_NAME_MAIN,
                coreset_ratio=CORESET_RATIO,
                aug_strength=meta["aug_strength"] if meta["representation_source"] == "simclr" else None)}

# Run one method on one category and return the row used later in the final tables.
def run_one_method_one_category(cat, method_name):
    runtime = build_method_runtime(cat, method_name)
    res = eval_method(runtime["test_loader"], runtime["score_fn"])

    return {
        "category": cat,
        "method": method_name,
        "run_mode": RUN_MODE,
        "representation_source": runtime["representation_source"],
        "anomaly_head": runtime["anomaly_head"],
        "aug_strength": runtime["aug_strength"],
        "backbone_name": BACKBONE_NAME if runtime["anomaly_head"] != "autoencoder" else "small_conv_ae",
        "layers": PATCH_LAYER_NAME_MAIN if runtime["anomaly_head"] != "autoencoder" else "reconstruction_error",
        "img_size": IMG_SIZE,
        "coreset_ratio": CORESET_RATIO if runtime["anomaly_head"] == "patchcore" else np.nan,
        "feature_dim": runtime["feature_dim"],
        "memory_bank_n": runtime["memory_bank_n"],
        "memory_bank_mb": runtime["memory_bank_mb"],
        "checkpoint_size_mb": runtime["checkpoint_size_mb"],
        "fit_sec": runtime["fit_sec"],
        "peak_gpu_mem_mb": runtime["peak_gpu_mem_mb"],
        **res}

# Collect one image-level score per item for threshold analysis.
def collect_image_scores(loader, score_fn):
    rows = []
    for x, y, _, paths in loader:
        scores, _ = score_fn(x)
        for i in range(len(paths)):
            rows.append({"path": paths[i], "label": int(y[i].item()), "score": float(scores[i])})
    return pd.DataFrame(rows)

# Keep the image, mask, and heatmap so the final failure figure can be built later.
def collect_detailed_preds(loader, score_fn):
    rows = []
    for x, y, m, paths in loader:
        scores, heat = score_fn(x)
        for i in range(len(paths)):
            rows.append({
                "path": paths[i],
                "label": int(y[i].item()),
                "score": float(scores[i]),
                "img_tensor": x[i].detach().cpu(),
                "mask": m[i].squeeze(0).numpy(),
                "heat": heat[i]
            })
    return rows

In [ ]:
#------------------------------------------------------------------------------
# 9.2 Main fair comparison
#------------------------------------------------------------------------------
# The point here is to compare methods under matched downstream settings.
MAIN_SSL_METHODS = [f"simclr_{tag}_patchcore" for tag in AVAILABLE_SSL_TAGS] + [f"simclr_{tag}_padim" for tag in AVAILABLE_SSL_TAGS]
rows_ssl_main = []

print_banner("Running main SSL methods with matched downstream settings")
print("Headline layer        :", PATCH_LAYER_NAME_MAIN)
print("Headline coreset ratio:", CORESET_RATIO)
print("SSL methods           :", MAIN_SSL_METHODS)

# Run the matched final comparison for each category.
for cat in CATEGORIES:
    print("\n" + "-" * 90)
    print("CATEGORY:", cat)
    print("-" * 90)
    for method_name in MAIN_SSL_METHODS:
        row = run_one_method_one_category(cat, method_name)
        rows_ssl_main.append(row)
        print(method_name, {k: row[k] for k in ["img_roc_auc", "img_pr_auc", "px_roc_auc", "fit_sec", "sec_per_img"]})

df_ssl_main = pd.DataFrame(rows_ssl_main)

baseline_keep = ["imagenet_patchcore", "imagenet_padim", "autoencoder"]
# Keep the per-category baseline rows so they can be merged with the SSL rows.
df_base_core = table_baselines[(table_baselines["category"] != "MEAN") & (table_baselines["method"].isin(baseline_keep))].copy()
df_base_core = df_base_core[df_base_core["category"].isin(CATEGORIES)].copy()

df_main_core = pd.concat([df_base_core, df_ssl_main], ignore_index=True)

mean_cols = ["img_roc_auc", "img_pr_auc", "px_roc_auc","fit_sec", "sec_per_img", "feature_dim","memory_bank_n", "memory_bank_mb", "checkpoint_size_mb", "peak_gpu_mem_mb"]
mean_df_main = (df_main_core.groupby(["method", "representation_source", "anomaly_head", "aug_strength"], as_index=False)[mean_cols].mean(numeric_only=True))
mean_df_main["category"] = "MEAN"
mean_df_main["run_mode"] = RUN_MODE
mean_df_main["backbone_name"] = ""
mean_df_main["layers"] = PATCH_LAYER_NAME_MAIN
mean_df_main["img_size"] = IMG_SIZE
mean_df_main["coreset_ratio"] = mean_df_main["anomaly_head"].map({"patchcore": CORESET_RATIO, "padim": np.nan, "autoencoder": np.nan})
mean_df_main["n_eval_images"] = np.nan

ordered_cols_main = ["category", "method", "run_mode", "representation_source", "anomaly_head", "aug_strength","backbone_name", "layers", "img_size", "coreset_ratio",
                     "img_roc_auc", "img_pr_auc", "px_roc_auc", "fit_sec", "sec_per_img", "n_eval_images","feature_dim", "memory_bank_n", "memory_bank_mb", "checkpoint_size_mb", "peak_gpu_mem_mb"]

table_main = pd.concat([df_main_core[ordered_cols_main], mean_df_main[ordered_cols_main]], ignore_index=True)
table_main.to_csv(TABLE_MAIN_PATH, index=False)

table_efficiency = mean_df_main[["method", "representation_source", "anomaly_head", "aug_strength","img_roc_auc", "img_pr_auc", "px_roc_auc",
                                 "fit_sec", "sec_per_img", "feature_dim", "memory_bank_n","memory_bank_mb", "checkpoint_size_mb", "peak_gpu_mem_mb"
                                 ]].sort_values(["img_pr_auc", "img_roc_auc"], ascending=False)
table_efficiency.to_csv(TABLE_EFFICIENCY_PATH, index=False)

print_banner("Main results")
display(mean_df_main.sort_values("img_pr_auc", ascending=False))
print("Saved:", TABLE_MAIN_PATH)
print("Saved:", TABLE_EFFICIENCY_PATH)

METHOD_ORDER = ["imagenet_patchcore", "imagenet_padim", "autoencoder","simclr_mild_patchcore", "simclr_strong_patchcore","simclr_mild_padim", "simclr_strong_padim"]
plot_df = mean_df_main.copy()
plot_df["method"] = pd.Categorical(plot_df["method"], categories=METHOD_ORDER, ordered=True)
plot_df = plot_df.sort_values("method")

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_specs = [
    ("img_roc_auc", "Mean image ROC-AUC"),
    ("img_pr_auc", "Mean image PR-AUC"),
    ("px_roc_auc", "Mean pixel ROC-AUC"),
    ("sec_per_img", "Mean inference sec / image")]
for ax, (metric, title) in zip(axes.ravel(), plot_specs):
    use_df = plot_df.dropna(subset=[metric])
    ax.bar(use_df["method"].astype(str), use_df[metric])
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=25)
    if "auc" in metric:
        ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(FIG_MAIN_PATH, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", FIG_MAIN_PATH)

In [ ]:
#------------------------------------------------------------------------------
# 9.3 Threshold analysis on the best deployable method
#------------------------------------------------------------------------------
# Thresholds are set from validation-good scores, then evaluated on the test split.
rank_main = mean_df_main.dropna(subset=["img_pr_auc", "img_roc_auc"], how="all").copy()
BEST_OVERALL_METHOD = rank_main.sort_values(["img_pr_auc", "img_roc_auc", "px_roc_auc"], ascending=False).iloc[0]["method"]

ssl_mean = mean_df_main[mean_df_main["representation_source"] == "simclr"].copy()
ssl_rank = ssl_mean.dropna(subset=["img_pr_auc", "img_roc_auc"], how="all").copy()
BEST_SSL_METHOD = None if len(ssl_rank) == 0 else ssl_rank.sort_values(["img_pr_auc", "img_roc_auc", "px_roc_auc"], ascending=False).iloc[0]["method"]

threshold_targets = [{"target_name": "best_overall", "method": BEST_OVERALL_METHOD}]
if BEST_SSL_METHOD is not None and BEST_SSL_METHOD != BEST_OVERALL_METHOD:
    threshold_targets.append({"target_name": "best_ssl", "method": BEST_SSL_METHOD})

print_banner("Threshold-analysis targets")
print("BEST_OVERALL_METHOD:", BEST_OVERALL_METHOD)
print("BEST_SSL_METHOD    :", BEST_SSL_METHOD)

threshold_rows = []
# Evaluate each chosen target method under the threshold policies.
for target in threshold_targets:
    target_name, method_name = target["target_name"], target["method"]
    for cat in CATEGORIES:
        runtime = build_method_runtime(cat, method_name)
        df_val = collect_image_scores(runtime["val_loader"], runtime["score_fn"])
        df_test = collect_image_scores(runtime["test_loader"], runtime["score_fn"])

        val_scores = df_val["score"].values.astype(float)
        y_true = df_test["label"].values.astype(int)
        y_score = df_test["score"].values.astype(float)

        for policy_name, q in THRESHOLD_POLICIES.items():
            thr = float(np.quantile(val_scores, q))
            mets = threshold_metrics(y_true, y_score, thr)
            threshold_rows.append({
                "target_name": target_name, "method": method_name, "category": cat,
                "policy": policy_name, "quantile": q, "threshold": thr, **mets})

df_thresholds = pd.DataFrame(threshold_rows)
mean_thresholds = (df_thresholds.groupby(["target_name", "method", "policy"], as_index=False)[["precision", "recall", "f1", "fpr"]].mean(numeric_only=True))
mean_thresholds["category"] = "MEAN"
mean_thresholds["quantile"] = np.nan
mean_thresholds["threshold"] = np.nan

table_thresholds = pd.concat([df_thresholds, mean_thresholds], ignore_index=True)
table_thresholds.to_csv(TABLE_THRESHOLDS_PATH, index=False)

print_banner("Threshold summary")
display(table_thresholds.tail(20))
print("Saved:", TABLE_THRESHOLDS_PATH)

In [ ]:
#------------------------------------------------------------------------------
# 9.4 Dense threshold sweep
#------------------------------------------------------------------------------
print_banner("Dense threshold sweep")

SWEEP_QUANTILES = np.round(np.linspace(0.90, 0.999, 20), 3)
sweep_rows = []

# Run a denser threshold sweep for smoother trade-off plots.
for target in threshold_targets:
    target_name = target["target_name"]
    method_name = target["method"]

    for cat in CATEGORIES:
        runtime = build_method_runtime(cat, method_name)
        df_val = collect_image_scores(runtime["val_loader"], runtime["score_fn"])
        df_test = collect_image_scores(runtime["test_loader"], runtime["score_fn"])

        val_scores = df_val["score"].values.astype(float)
        y_true = df_test["label"].values.astype(int)
        y_score = df_test["score"].values.astype(float)

        for q in SWEEP_QUANTILES:
            thr = float(np.quantile(val_scores, q))
            mets = threshold_metrics(y_true, y_score, thr)

            sweep_rows.append({
                "target_name": target_name,
                "method": method_name,
                "category": cat,
                "quantile": q,
                "threshold": thr,
                **mets})

df_threshold_sweep = pd.DataFrame(sweep_rows)

mean_threshold_sweep = (df_threshold_sweep.groupby(["target_name", "method", "quantile"], as_index=False)[["precision", "recall", "f1", "fpr"]].mean(numeric_only=True))
mean_threshold_sweep["category"] = "MEAN"
mean_threshold_sweep["threshold"] = np.nan

table_threshold_sweep = pd.concat([df_threshold_sweep, mean_threshold_sweep], ignore_index=True)
table_threshold_sweep.to_csv(TABLE_THRESHOLD_SWEEP_PATH, index=False)

print("Saved:", TABLE_THRESHOLD_SWEEP_PATH)
display(table_threshold_sweep.tail(20))

In [ ]:
#------------------------------------------------------------------------------
# 9.5 Ablation study
#------------------------------------------------------------------------------
# Reuse the same evaluation code while changing one setting at a time.
ABLATION_SSL_TAG = "strong" if "strong" in AVAILABLE_SSL_TAGS else AVAILABLE_SSL_TAGS[0]

def run_ssl_ablation_category(cat, aug_strength, anomaly_head, layer_name, coreset_ratio):
    # Run one SSL-based setting on one category for the ablation table.
    runtime = build_feature_method_runtime(
        cat=cat, representation_source="simclr", anomaly_head=anomaly_head,
        layer_name=layer_name, coreset_ratio=coreset_ratio, aug_strength=aug_strength)
    res = eval_method(runtime["test_loader"], runtime["score_fn"])
    return {
        "category": cat, "representation_source": "simclr", "anomaly_head": anomaly_head,
        "aug_strength": aug_strength, "layer_name": layer_name, "coreset_ratio": coreset_ratio,
        "feature_dim": runtime["feature_dim"], "memory_bank_n": runtime["memory_bank_n"],
        "memory_bank_mb": runtime["memory_bank_mb"], "checkpoint_size_mb": runtime["checkpoint_size_mb"],
        "fit_sec": runtime["fit_sec"], "peak_gpu_mem_mb": runtime["peak_gpu_mem_mb"], **res}

ablation_rows = []

# A. augmentation strength
print_banner("Ablation A: augmentation strength")
# Change augmentation strength while keeping the rest of the setup fixed.
for aug_strength in AVAILABLE_SSL_TAGS:
    for cat in CATEGORIES:
        ablation_rows.append({"ablation_factor": "augmentation_strength", "setting": aug_strength,
            **run_ssl_ablation_category(cat, aug_strength, "patchcore", PATCH_LAYER_NAME_MAIN, CORESET_RATIO)})
    for cat in CATEGORIES:
        ablation_rows.append({
            "ablation_factor": "augmentation_strength_padim", "setting": aug_strength,
            **run_ssl_ablation_category(cat, aug_strength, "padim", PATCH_LAYER_NAME_MAIN, np.nan)})
# B. layer
print_banner("Ablation B: embedding layer")
for layer_name in LAYER_ABL_VALUES:
    for cat in CATEGORIES:
        ablation_rows.append({"ablation_factor": "embedding_layer", "setting": layer_name,
            **run_ssl_ablation_category(cat, ABLATION_SSL_TAG, "patchcore", layer_name, CORESET_RATIO)})
# C. coreset ratio
print_banner("Ablation C: coreset ratio")
for ratio in CORESET_ABL_VALUES:
    for cat in CATEGORIES:
        ablation_rows.append({"ablation_factor": "coreset_ratio", "setting": str(ratio),
            **run_ssl_ablation_category(cat, ABLATION_SSL_TAG, "patchcore", PATCH_LAYER_NAME_MAIN, ratio)})

df_ablation = pd.DataFrame(ablation_rows)
mean_ablation = (df_ablation.groupby(
    ["ablation_factor", "setting", "representation_source", "anomaly_head", "aug_strength", "layer_name", "coreset_ratio"],
        as_index=False)[["img_roc_auc", "img_pr_auc", "px_roc_auc", "fit_sec", "sec_per_img", "feature_dim",
       "memory_bank_n", "memory_bank_mb", "checkpoint_size_mb", "peak_gpu_mem_mb"]].mean(numeric_only=True))
mean_ablation["category"] = "MEAN"
table_ablation = pd.concat([df_ablation, mean_ablation], ignore_index=True)
table_ablation.to_csv(TABLE_ABLATION_PATH, index=False)
# Check that the ablation settings really changed what they were supposed to change.
print_banner("Ablation sanity checks")

aug_check = table_ablation[
    (table_ablation["category"] == "MEAN") &
    (table_ablation["ablation_factor"] == "augmentation_strength")][["setting", "img_pr_auc", "img_roc_auc", "px_roc_auc"]].copy()

coreset_check = table_ablation[
    (table_ablation["category"] == "MEAN") &
    (table_ablation["ablation_factor"] == "coreset_ratio")][["setting", "memory_bank_n", "memory_bank_mb", "img_pr_auc"]].copy()

layer_check = table_ablation[
    (table_ablation["category"] == "MEAN") &
    (table_ablation["ablation_factor"] == "embedding_layer")][["setting", "img_pr_auc", "img_roc_auc", "px_roc_auc"]].copy()

print("\nAugmentation-strength summary")
display(aug_check.sort_values("setting"))

print("\nCoreset-ratio summary")
display(coreset_check.sort_values("setting"))

print("\nLayer-ablation summary")
display(layer_check.sort_values("setting"))

if len(aug_check) < 2:
    print("Warning: augmentation ablation only contains one SSL setting.")
else:
    print("Good: augmentation ablation contains multiple SSL settings.")

if coreset_check["memory_bank_n"].nunique() == 1:
    print("Warning: all coreset settings produced the same memory bank size.")
else:
    print("Good: coreset settings produced different memory bank sizes.")
print_banner("Ablation summary")
display(table_ablation.tail(25))
print("Saved:", TABLE_ABLATION_PATH)

In [ ]:
#------------------------------------------------------------------------------
# 9.6 Failure analysis
#------------------------------------------------------------------------------
# Inspect the weakest category for the best overall method and save example failures.
failure_method = BEST_OVERALL_METHOD
best_overall_by_cat = table_main[(table_main["category"] != "MEAN") & (table_main["method"] == failure_method)].copy()
FAIL_CAT = best_overall_by_cat.sort_values("img_pr_auc", ascending=True).iloc[0]["category"]

thr_balanced = table_thresholds[
    (table_thresholds["target_name"] == "best_overall") &
    (table_thresholds["method"] == failure_method) &
    (table_thresholds["category"] == FAIL_CAT) &
    (table_thresholds["policy"] == "balanced")]["threshold"].iloc[0]

print_banner("Failure analysis selection")
print("Method            :", failure_method)
print("Category          :", FAIL_CAT)
print("Balanced threshold:", thr_balanced)

runtime = build_method_runtime(FAIL_CAT, failure_method)
detailed_rows = collect_detailed_preds(runtime["test_loader"], runtime["score_fn"])

# Turn the raw anomaly scores into predicted labels and error types.
for row in detailed_rows:
    row["pred"] = int(row["score"] >= thr_balanced)
    row["error_type"] = (
        "FP" if (row["label"] == 0 and row["pred"] == 1) else
        "FN" if (row["label"] == 1 and row["pred"] == 0) else
        "TP" if (row["label"] == 1 and row["pred"] == 1) else
        "TN")
    row["defect_type"] = Path(row["path"]).parent.name
    row["file_name"] = Path(row["path"]).name

df_failures = pd.DataFrame([{"method": failure_method, "category": FAIL_CAT, "path": r["path"], "file_name": r["file_name"],
                             "defect_type": r["defect_type"], "label": r["label"], "pred": r["pred"], "score": r["score"], "error_type": r["error_type"]} for r in detailed_rows])
df_failures.to_csv(TABLE_FAILURES_PATH, index=False)
print("Saved:", TABLE_FAILURES_PATH)

# Show a few confident examples so the qualitative failure figure is easy to read.
fps = sorted([r for r in detailed_rows if r["error_type"] == "FP"], key=lambda r: r["score"], reverse=True)[:2]
fns = sorted([r for r in detailed_rows if r["error_type"] == "FN"], key=lambda r: r["score"])[:2]
tps = sorted([r for r in detailed_rows if r["error_type"] == "TP"], key=lambda r: r["score"], reverse=True)[:1]
examples = tps + fps + fns
if len(examples) == 0:
    anomalies = [r for r in detailed_rows if r["label"] == 1]
    examples = sorted(anomalies, key=lambda r: r["score"], reverse=True)[:4]

n = len(examples)
plt.figure(figsize=(12, max(3, 3.0 * n)))
for i, row in enumerate(examples):
    caption = f"{row['error_type']} | {row['defect_type']} | score={row['score']:.3f}"

    ax1 = plt.subplot(n, 3, i * 3 + 1)
    overlay(ax1, row["img_tensor"], None, title="Image")
    ax1.set_ylabel(caption, fontsize=9)

    ax2 = plt.subplot(n, 3, i * 3 + 2)
    show_mask(ax2, row["mask"], title="GT mask")

    ax3 = plt.subplot(n, 3, i * 3 + 3)
    overlay(ax3, row["img_tensor"], row["heat"], title="Heatmap")

plt.tight_layout()
plt.savefig(FIG_FAILURES_PATH, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", FIG_FAILURES_PATH)

# 10.0 Final visualisations

## Purpose
- turn the saved result tables into clear figures that are easier to discuss in the final report
- compare methods from several perspectives, including average performance, category-level consistency, efficiency, threshold trade-offs, and ablation behaviour
- highlight patterns that are harder to see from raw tables alone, such as which methods are consistently strong across categories and which settings create practical trade-offs
- save report-ready figures so the final write-up can focus on interpretation rather than figure generation

## Process
1. load the main result tables needed for plotting.
2. build category-level and mean-level comparison views.
3. generate summary figures for performance, efficiency, thresholds, ablations, and failures.
4. save the final figures for later use in the report.

In [ ]:
#------------------------------------------------------------------------------
# 10.1 Final visualisation block
#------------------------------------------------------------------------------
# Build cleaner report figures from the saved result tables.
print_banner("Final visualisation block")

#------------------------------------------------------------------------------
# Load tables for visualisations
#------------------------------------------------------------------------------
df_main_viz = table_main.copy()
df_abl_viz = table_ablation.copy()
df_thr_viz = table_thresholds.copy()
df_fail_viz = df_failures.copy()

# Load the dense threshold sweep if it exists, otherwise just use the named thresholds.
df_thr_sweep_viz = None
if "table_threshold_sweep" in globals():
    df_thr_sweep_viz = table_threshold_sweep.copy()
elif "TABLE_THRESHOLD_SWEEP_PATH" in globals() and Path(TABLE_THRESHOLD_SWEEP_PATH).exists():
    df_thr_sweep_viz = pd.read_csv(TABLE_THRESHOLD_SWEEP_PATH)

#------------------------------------------------------------------------------
# Figure output paths
#------------------------------------------------------------------------------
FIG_HEATMAP_PATH = RESULTS_DIR / "fig_final_heatmap_pr_auc.png"
FIG_RANKS_PATH = RESULTS_DIR / "fig_final_ranks_and_wins.png"
FIG_ACC_EFF_PATH = RESULTS_DIR / "fig_final_accuracy_vs_efficiency.png"
FIG_THRESH_TRADEOFF_PATH = RESULTS_DIR / "fig_final_threshold_tradeoff.png"
FIG_ABL_AUG_PATH = RESULTS_DIR / "fig_final_ablation_augmentation.png"
FIG_ABL_LAYER_PATH = RESULTS_DIR / "fig_final_ablation_layer.png"
FIG_ABL_CORESET_PATH = RESULTS_DIR / "fig_final_ablation_coreset.png"
FIG_FAILURE_COUNTS_PATH = RESULTS_DIR / "fig_final_failure_counts.png"

#------------------------------------------------------------------------------
# Small helpers
#------------------------------------------------------------------------------
# Save and show each figure in the same way.
def save_show(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=240, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

# Use cleaner method names in the report figures.
def method_display_name(method):
    name_map = {
        "imagenet_patchcore": "ImageNet PatchCore",
        "imagenet_padim": "ImageNet PaDiM",
        "autoencoder": "Autoencoder",
        "simclr_mild_patchcore": "SimCLR Mild PatchCore",
        "simclr_strong_patchcore": "SimCLR Strong PatchCore",
        "simclr_mild_padim": "SimCLR Mild PaDiM",
        "simclr_strong_padim": "SimCLR Strong PaDiM",}
    return name_map.get(method, method)

# Clean up anomaly-head labels in the ablation plots.
def head_display_name(head):
    if pd.isna(head):
        return "Unknown"
    head = str(head).strip().lower()
    if "patch" in head:
        return "PatchCore"
    if "padim" in head:
        return "PaDiM"
    if "auto" in head:
        return "Autoencoder"
    return str(head)

METHOD_ORDER = [
    "imagenet_patchcore",
    "imagenet_padim",
    "autoencoder",
    "simclr_mild_patchcore",
    "simclr_strong_patchcore",
    "simclr_mild_padim",
    "simclr_strong_padim",]

METHOD_ORDER_PRESENT = [m for m in METHOD_ORDER if m in set(df_main_viz["method"].unique())]
PRIMARY_METRIC = "img_pr_auc"

#------------------------------------------------------------------------------
# 10.1 Category x method heatmap using image PR-AUC
#------------------------------------------------------------------------------
# Make the heatmap larger so it is readable in the report.
df_heat = df_main_viz[df_main_viz["category"] != "MEAN"].copy()
heat = (df_heat.pivot_table(index="category", columns="method", values=PRIMARY_METRIC, aggfunc="mean")
           .reindex(columns=METHOD_ORDER_PRESENT)
           .sort_index())

fig, ax = plt.subplots(figsize=(14, 8.5))
im = ax.imshow(heat.values, aspect="auto", vmin=0, vmax=1)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Image PR-AUC")

ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels([method_display_name(c) for c in heat.columns], rotation=30, ha="right", fontsize=10)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=10)
ax.set_title("Category-level image PR-AUC heatmap", fontsize=14)

# Write the PR-AUC value inside each cell.
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        if pd.notna(val):
            txt_color = "white" if val < 0.58 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=txt_color)

save_show(fig, FIG_HEATMAP_PATH)

#------------------------------------------------------------------------------
# 10.2 Average rank and category wins
#------------------------------------------------------------------------------
# Use horizontal bars so the method names are easier to read.
rank_df = heat.rank(axis=1, ascending=False, method="average")
avg_rank = rank_df.mean(axis=0).sort_values()
wins = (rank_df == 1).sum(axis=0).reindex(avg_rank.index)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
y = np.arange(len(avg_rank))

axes[0].barh(y, avg_rank.values)
axes[0].set_yticks(y)
axes[0].set_yticklabels([method_display_name(x) for x in avg_rank.index], fontsize=10)
axes[0].invert_yaxis()
axes[0].set_xlabel("Average rank")
axes[0].set_title("Average category rank")
axes[0].grid(True, axis="x", alpha=0.3)

for i, v in enumerate(avg_rank.values):
    axes[0].text(v + 0.03, i, f"{v:.2f}", va="center", fontsize=9)

axes[1].barh(y, wins.values)
axes[1].set_yticks(y)
axes[1].set_yticklabels([method_display_name(x) for x in wins.index], fontsize=10)
axes[1].invert_yaxis()
axes[1].set_xlabel("Number of wins")
axes[1].set_title("Category wins")
axes[1].grid(True, axis="x", alpha=0.3)

for i, v in enumerate(wins.values):
    axes[1].text(v + 0.10, i, f"{int(v)}", va="center", fontsize=9)

save_show(fig, FIG_RANKS_PATH)

#------------------------------------------------------------------------------
# 10.3 Accuracy vs efficiency scatter using MEAN rows
#------------------------------------------------------------------------------
# Keep the same figure idea but make labels easier to read.
df_mean = df_main_viz[df_main_viz["category"] == "MEAN"].copy()
df_mean = df_mean.drop_duplicates(subset=["method"]).copy()
df_mean = df_mean[df_mean["method"].isin(METHOD_ORDER_PRESENT)].copy()
df_mean["label"] = df_mean["method"].map(method_display_name)

size_source = df_mean["memory_bank_mb"].fillna(df_mean["checkpoint_size_mb"]).fillna(1.0).astype(float)
sizes = 220 + 100 * (size_source / max(size_source.max(), 1e-8))

fig, ax = plt.subplots(figsize=(8.5, 6))
ax.scatter(df_mean["sec_per_img"], df_mean[PRIMARY_METRIC], s=sizes, alpha=0.8)

df_mean_plot = df_mean.sort_values([PRIMARY_METRIC, "sec_per_img"], ascending=[False, True]).reset_index(drop=True)

for i, row in df_mean_plot.iterrows():
    dx = 8 if i % 2 == 0 else -8
    dy = 6 if (i // 2) % 2 == 0 else -10
    ha = "left" if dx > 0 else "right"
    ax.annotate(
        row["label"],
        (row["sec_per_img"], row[PRIMARY_METRIC]),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=9,
        ha=ha)

ax.set_xlabel("Inference sec / image")
ax.set_ylabel("Image PR-AUC")
ax.set_title("Accuracy vs efficiency")
ax.grid(True, alpha=0.3)
save_show(fig, FIG_ACC_EFF_PATH)

#------------------------------------------------------------------------------
# 10.4 Threshold trade-off for best overall and best SSL methods
#------------------------------------------------------------------------------
# Plot the named threshold policies and, if available, overlay a smoother sweep line.
df_thr_mean = df_thr_viz[df_thr_viz["category"] == "MEAN"].copy()

if len(df_thr_mean) > 0:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    targets = df_thr_mean[["target_name", "method"]].drop_duplicates()

    for _, target_row in targets.iterrows():
        target_name = target_row["target_name"]
        method = target_row["method"]

        use_df = df_thr_mean[
            (df_thr_mean["target_name"] == target_name) &
            (df_thr_mean["method"] == method)
        ].sort_values("fpr")

        # Add a smoother operating curve if the dense sweep table is available.
        if df_thr_sweep_viz is not None:
            sweep_df = df_thr_sweep_viz.copy()
            if "category" in sweep_df.columns:
                sweep_df = sweep_df[sweep_df["category"] == "MEAN"].copy()
            if "target_name" in sweep_df.columns:
                sweep_df = sweep_df[sweep_df["target_name"] == target_name].copy()
            if "method" in sweep_df.columns:
                sweep_df = sweep_df[sweep_df["method"] == method].copy()
            if len(sweep_df) > 1 and {"fpr", "recall"}.issubset(sweep_df.columns):
                sweep_df = sweep_df.sort_values("fpr")
                ax.plot(
                    sweep_df["fpr"],
                    sweep_df["recall"],
                    linewidth=1.6,
                    alpha=0.35)

        ax.plot(
            use_df["fpr"],
            use_df["recall"],
            marker="o",
            linewidth=2.2,
            label=f"{target_name}: {method_display_name(method)}")

        for _, r in use_df.iterrows():
            ax.annotate(
                r["policy"],
                (r["fpr"], r["recall"]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8)

    ax.set_xlabel("False positive rate")
    ax.set_ylabel("Recall")
    ax.set_title("Threshold trade-off")
    ax.grid(True, alpha=0.3)
    ax.legend()
    save_show(fig, FIG_THRESH_TRADEOFF_PATH)

#------------------------------------------------------------------------------
# 10.5 Augmentation-strength ablation
#------------------------------------------------------------------------------
# Show mild vs strong clearly, and support both PatchCore and PaDiM rows.
df_abl_mean = df_abl_viz[df_abl_viz["category"] == "MEAN"].copy()

aug_df = df_abl_mean[df_abl_mean["ablation_factor"].isin(["augmentation_strength", "augmentation_strength_padim"])].copy()

if len(aug_df) > 0:
    if "anomaly_head" not in aug_df.columns:
        aug_df["anomaly_head"] = np.where(
            aug_df["ablation_factor"] == "augmentation_strength_padim", "padim", "patchcore")

    aug_df["head_label"] = aug_df["anomaly_head"].map(head_display_name)
    aug_df["setting"] = aug_df["setting"].astype(str).str.lower()

    strength_order = [x for x in ["mild", "strong"] if x in set(aug_df["setting"])]
    head_order = [x for x in ["PatchCore", "PaDiM"] if x in set(aug_df["head_label"])]

    x = np.arange(len(head_order))
    width = 0.34 if len(strength_order) <= 2 else 0.8 / max(len(strength_order), 1)

    fig, ax = plt.subplots(figsize=(8, 5.5))

    for i, strength in enumerate(strength_order):
        vals = []
        for head in head_order:
            row = aug_df[(aug_df["head_label"] == head) & (aug_df["setting"] == strength)]
            vals.append(float(row[PRIMARY_METRIC].iloc[0]) if len(row) else np.nan)

        xpos = x + (i - (len(strength_order) - 1) / 2) * width
        bars = ax.bar(xpos, vals, width=width, label=strength.title())

        for bar, val in zip(bars, vals):
            if pd.notna(val):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    val + 0.003,
                    f"{val:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(head_order)
    ax.set_ylabel("Image PR-AUC")
    ax.set_title("Augmentation-strength ablation")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(title="Strength")
    save_show(fig, FIG_ABL_AUG_PATH)

#------------------------------------------------------------------------------
# 10.6 Layer ablation
#------------------------------------------------------------------------------
# Keep the layer plot simple and add values to each point.
layer_df = df_abl_mean[df_abl_mean["ablation_factor"] == "embedding_layer"].copy()

if len(layer_df) > 0:
    layer_df["layer_num"] = layer_df["setting"].str.replace("layer", "", regex=False).astype(int)
    layer_df = layer_df.sort_values("layer_num")

    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.plot(layer_df["setting"], layer_df[PRIMARY_METRIC], marker="o", linewidth=2)

    for _, row in layer_df.iterrows():
        ax.text(row["setting"], row[PRIMARY_METRIC] + 0.0015, f"{row[PRIMARY_METRIC]:.3f}", ha="center", fontsize=9)

    ax.set_xlabel("Embedding layer")
    ax.set_ylabel("Image PR-AUC")
    ax.set_title("PatchCore layer ablation")
    ax.grid(True, alpha=0.3)
    save_show(fig, FIG_ABL_LAYER_PATH)

#------------------------------------------------------------------------------
# 10.7 Coreset ablation
#------------------------------------------------------------------------------
# Show both the accuracy change and the memory-bank growth together.
coreset_df = df_abl_mean[df_abl_mean["ablation_factor"] == "coreset_ratio"].copy()

if len(coreset_df) > 0:
    coreset_df["ratio_num"] = coreset_df["setting"].astype(float)
    coreset_df = coreset_df.sort_values("ratio_num")

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))

    axes[0].plot(coreset_df["ratio_num"], coreset_df[PRIMARY_METRIC], marker="o", linewidth=2)
    for _, row in coreset_df.iterrows():
        axes[0].text(row["ratio_num"], row[PRIMARY_METRIC] + 0.0015, f"{row[PRIMARY_METRIC]:.3f}", ha="center", fontsize=9)
    axes[0].set_xlabel("Coreset ratio")
    axes[0].set_ylabel("Image PR-AUC")
    axes[0].set_title("Coreset ratio vs performance")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(coreset_df["ratio_num"], coreset_df["memory_bank_n"], marker="o", linewidth=2)
    for _, row in coreset_df.iterrows():
        axes[1].text(row["ratio_num"], row["memory_bank_n"] + max(5, 0.01 * coreset_df["memory_bank_n"].max()), f"{int(row['memory_bank_n'])}", ha="center", fontsize=9)
    axes[1].set_xlabel("Coreset ratio")
    axes[1].set_ylabel("Memory bank size")
    axes[1].set_title("Coreset ratio vs memory bank size")
    axes[1].grid(True, alpha=0.3)

    save_show(fig, FIG_ABL_CORESET_PATH)

#------------------------------------------------------------------------------
# 10.8 Failure-type count plot
#------------------------------------------------------------------------------
# Add count labels so this figure can stand on its own in the report.
if len(df_fail_viz) > 0 and "error_type" in df_fail_viz.columns:
    fail_counts = (
        df_fail_viz["error_type"]
        .value_counts()
        .reindex(["TP", "FP", "FN", "TN"])
        .fillna(0)
        .astype(int))

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    bars = ax.bar(fail_counts.index, fail_counts.values)

    for bar, val in zip(bars, fail_counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + max(0.5, 0.02 * max(fail_counts.values.max(), 1)),
            f"{int(val)}",
            ha="center",
            va="bottom",
            fontsize=9)

    ax.set_xlabel("Outcome type")
    ax.set_ylabel("Count")
    ax.set_title("Failure-analysis outcome counts")
    ax.grid(True, axis="y", alpha=0.3)
    save_show(fig, FIG_FAILURE_COUNTS_PATH)

print_banner("Final visualisation block complete")
print("Saved figures:")
for p in [
    FIG_HEATMAP_PATH,
    FIG_RANKS_PATH,
    FIG_ACC_EFF_PATH,
    FIG_THRESH_TRADEOFF_PATH,
    FIG_ABL_AUG_PATH,
    FIG_ABL_LAYER_PATH,
    FIG_ABL_CORESET_PATH,
    FIG_FAILURE_COUNTS_PATH,]:
    if Path(p).exists():
        print(" -", p)

# 11.0 Top 5 Key findings

1. **ImageNet PaDiM was the strongest overall method.** It achieved the best average image-level and pixel-level performance across the tested categories, making it the most reliable method in this study.

2. **The SimCLR-based methods did not outperform the ImageNet baselines.** Although the self-supervised models trained successfully and produced reasonable results, the pretrained ImageNet feature methods remained stronger overall.

3. **Mild SimCLR augmentation performed slightly better than strong augmentation.** This suggests that overly aggressive contrastive augmentation may reduce downstream anomaly detection quality in this industrial setting.

4. **PatchCore performance improved as the coreset ratio increased.** Keeping a larger memory bank helped image PR-AUC, although this also increased memory bank size and reduced efficiency.

5. **Threshold choice changed the practical behaviour of the models.** Lower thresholds increased recall but also increased false alarms, showing that operating point selection is important when balancing defect detection against unnecessary alerts.

In [ ]:
#------------------------------------------------------------------------------
# Zip the full working project so it can be downloaded in one file
#------------------------------------------------------------------------------
import shutil

PROJECT_DIR = WORK_ROOT
ZIP_BASE_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
ZIP_BASE = ZIP_BASE_DIR / f"{WORK_ROOT.name}_{RUN_MODE}_bundle"

if PROJECT_DIR.exists():
    shutil.make_archive(str(ZIP_BASE), "zip", root_dir=PROJECT_DIR)
    print("Saved:", str(ZIP_BASE) + ".zip")
else:
    print("Project directory does not exist, so no zip file was created:", PROJECT_DIR)
